# The International Brain Laboratory (2021) on DANDI:000045

Paper: The International Brain Laboratory, "Standardized and reproducible measurement of
decision-making in mice", eLife 10:e63711 (2021), doi:10.7554/eLife.63711.

Dataset: DANDI:000045, "IBL behavioral data", version **0.211209.1413**
(doi:10.48324/dandi.000045/0.211209.1413). The paper cites no version of the dandiset (its data
statement points to data.internationalbrainlab.org, and the dandiset names the bioRxiv preprint).
There are two published versions, 0.211209.1412 and 0.211209.1413, made a minute apart; they hold
the same files (checked below), so the most recent is pinned.

Claims: `claims/000045/10.7554-elife.63711.yaml`, written from the version of record (eLife,
PMC8137147) with its Methods, figure legends and Appendix 1, and the paper's code
(github.com/int-brain-lab/paper-behavior at commit edc4531). The claims file decides what is tested
and how; this notebook does not change it.

How claims are judged. The questions this attempt asked were answered on 2026-10-22
(`questions/000045.yaml`), and the claims file now names a main reading and variants for basic
proficiency (P1), the first full-task session (P1b), 50:50 sessions and the classifier's subsample
size. The main reading decides each outcome; the variants are shown, and the attempt record says
where one gives a different outcome. Where a claim's criterion lists readings of its own (sessions,
filters, accuracy measures), a claim matches when it matches under any of them, unless the criterion
says otherwise or the readings are of a detail the paper does not fix: there they must agree, or
the claim is blocked as `methods_underspecified`. Where the expected values are not all met but
some are, or the effect holds without the reported strength, the outcome is
`partially_reproduced`.

| Claim | Tested here |
|---|---|
| cohort-counts | tested |
| fig2g-days-to-proficiency | tested |
| fig2g-days-differ-across-labs | tested |
| fig2s1-trials-to-proficiency | tested |
| fig2h-proportion-learned-40-days | tested |
| fig2bc-threshold-bias-iqr | tested |
| fig2-abs-bias-threshold-during-training | tested |
| fig2s2-variability-decreases | tested |
| early-sessions-predict-slow-learners | tested; blocked, the criterion's readings disagree |
| fig3c-easy-errors-basic | tested |
| fig3de-threshold-bias-basic | tested |
| fig3cde-no-lab-difference-medians | tested |
| fig3cde-no-lab-difference-variances | tested |
| fig3s2a-trial-duration | tested |
| fig3s2b-trials-per-session | tested |
| fig3f-classifier-chance-basic | tested |
| fig3f-positive-control | tested |
| fig3s3-other-classifiers | tested in part; blocked, the Random Forest cannot run 2000 subsamples in the time limit |
| lab-link (added: the labs and their time zones) | tested |
| fig3s1a-choice-consistency-basic | tested |
| fig3s1bc-choice-consistency-full | tested |
| fig3s4-looser-criterion | tested |
| fig3s5-first-full-sessions | tested |
| fig3s5a-prior-used-from-first-sessions | tested |
| full-task-training-time | tested |
| fig4cd-biasshift-largest-at-zero | tested |
| fig4de-biasshift-no-lab-difference | tested |
| fig4f-threshold-unchanged-by-block | tested |
| fig4gh-lapses-change-with-block | tested |
| fig4i-bias-shifts-with-block | tested |
| fig4fi-no-lab-difference | tested |
| fig4j-classifier-chance-full | tested |
| fig5cf-glm-weights | tested |
| fig5dg-history-weights-higher-full | tested |
| fig5s2-glm-accuracy | tested; blocked, the two readings of the 50% weight disagree |
| fig5s2-weights-similar-across-labs | tested |
| fig5s1b-win-stay-lose-switch | tested |
| fig5s1c-updating-by-contrast | tested |
| session-end-criteria | tested |

The notebook streams every file (about six minutes) and runs in about twenty-five minutes in all.

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")
import time
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import ibl_behavior as ib

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
T0 = time.time()
print("dandiset", ib.DANDISET_ID, "version", ib.DANDISET_VERSION)

## Loading

Every file of the pinned version is streamed; only the trials table, the session metadata and
the rig settings are read. This takes about six minutes.

In [ ]:
#LOAD
meta = ib.version_metadata()
assets = ib.list_assets()
other = ib.list_assets("0.211209.1412")
print(meta["doi"], "| assets", len(assets), "| subjects (archive summary)",
      meta["assetsSummary"]["numberOfSubjects"])
print("the two published versions hold the same files:",
      sorted(assets.blob) == sorted(other.blob) and sorted(assets.path) == sorted(other.path))
sessions = ib.read_all(assets.asset_id.tolist())
errors = [s for s in sessions if "error" in s]
print("files read", len(sessions), "| failed", len(errors))
sessions = [s for s in sessions if "error" not in s]
S = ib.session_table(sessions)
T = ib.trials_frame(sessions)
print(f"loaded in {(time.time() - T0) / 60:.1f} min")

## Data check

What was loaded, and each check of the description against the data.

In [ ]:
import hashlib
print("sessions", len(S), "| with trials", (S.n_trials > 0).sum(), "| trials", len(T))
print("subjects", S.subject_id.nunique(), "| nicknames", S.nickname.nunique(),
      "| subject folder = subject_id in file:", (assets.set_index("asset_id").subject_dir
                                                  .reindex(S.asset_id).to_numpy() == S.subject_id).all())
print("dates", S.date.min(), "to", S.date.max())
print("\nsessions by institution and lab number (code numbering by institution)")
print(S.groupby(["institution", "lab_number"]).agg(sessions=("session_id", "size"),
                                                    mice=("subject_id", "nunique")))
print("\nsessions by task protocol")
print(S.protocol.str.replace(r"[\d.]+$", "", regex=True).value_counts())
print("\nsessions by task as used here:", S.task.value_counts().to_dict())
print("sex of mice:", S.groupby("subject_id").sex.first().value_counts().to_dict())

# duplicated recordings: identical choices, contrasts and stimulus times
key = {}
for s in sessions:
    if s.get("choice") is None or len(s["choice"]) < 20:
        continue
    h = hashlib.md5(np.ascontiguousarray(s["choice"]).tobytes()
                    + np.ascontiguousarray(s["contrastLeft"]).tobytes()
                    + np.ascontiguousarray(s["stimOn_times"]).tobytes()).hexdigest()
    key.setdefault(h, []).append(s["session_id"])
dups = [v for v in key.values() if len(v) > 1]
print("\nrecordings deposited more than once:", len(dups))
for v in dups:
    print(S.set_index("session_id").loc[v, ["nickname", "start", "n_trials", "protocol"]])
dup_ids = {x for v in dups for x in v}
print("excluded session of the code present:", (S.session_id == ib.EXCLUDED_SESSION).any())

In [ ]:
# Trial variables: values, ranges and units, against the column descriptions
print("contrastLeft/Right values (fraction):", sorted(set(np.unique(T.contrastLeft.dropna()))
                                                     | set(np.unique(T.contrastRight.dropna()))))
print("trials with both or neither contrast set:",
      int((T.contrastLeft.notna() == T.contrastRight.notna()).sum()))
print("signed contrast (%) values:", sorted(T.signed_contrast.unique()))
odd = T[T.abs_contrast.isin([4.0, 8.0])].session_id.unique()
print("sessions with 4% or 8% contrasts:", S[S.session_id.isin(odd)][["nickname", "date", "protocol"]].to_dict("records"))
print("probabilityLeft values:", T.probabilityLeft.round(3).value_counts().to_dict())
tt = T.merge(S[["session_id", "task"]], on="session_id")
tb = tt[tt.task == "training"]
print("basic-task sessions with probabilityLeft other than 0.5 on some trial: %d of %d; trials %.1f%%"
      % ((tb.probabilityLeft != 0.5).groupby(tb.session_id).any().sum(), tb.session_id.nunique(),
         100 * (tb.probabilityLeft != 0.5).mean()))
print("basic task, fraction of left stimuli by probabilityLeft:",
      tb.groupby(tb.probabilityLeft.round(1)).contrastLeft.apply(lambda x: round(x.notna().mean(), 3)).to_dict())
del tt, tb
print("choice values:", T.choice.value_counts().to_dict(), "| feedbackType:", T.feedbackType.value_counts().to_dict())
print("no-response trials (choice 0): %.2f%%" % (100 * (T.choice == 0).mean()))
# choice convention: on trials with a stimulus on one side, which choice is rewarded
t = T[(T.choice != 0) & (T.abs_contrast > 0)]
print("\nrewarded fraction by stimulus side and choice:")
print(t.groupby([np.where(t.signed_contrast > 0, "right stim", "left stim"), "choice"]).correct.mean().round(4))
print("-> choice -1 (CCW) is the rightward choice, as coded in trials_frame")
q = T.rt.quantile([0, 0.01, 0.5, 0.99, 1]).round(3).to_dict()
print("\ntrial duration, response - stimOn (s), quantiles:", q)
print("negative trial durations:", int((T.rt < 0).sum()), "| missing stimOn:", int(T.stimOn_times.isna().sum()),
      "| missing response time:", int(T.response_times.isna().sum()))
miss = T.stimOn_times.isna().groupby(T.session_id).mean()
print("sessions with some trials lacking stimOn: %d; with none: %d" % ((miss > 0).sum(), (miss == 1).sum()))
print("sessions with a negative trial duration: %d" % (T.rt < 0).groupby(T.session_id).any().sum())
print("trials per session: median", S.n_trials.median(), "range", S.n_trials.min(), "-", S.n_trials.max())
print("session duration (min, end - start): median %.1f, range %.1f - %.1f, missing end %d"
      % (S.duration_min.median(), S.duration_min.min(), S.duration_min.max(), S.duration_min.isna().sum()))
# session_start_time is stated in UTC (+00:00); start hours by lab
hour = pd.to_datetime(S.start.str[:19]).dt.hour
print("\nsession_start_time offsets:", S.start.str[19:].value_counts().to_dict())
print("start hour by lab (5th, 50th, 95th percentile):")
print(S.assign(hour=hour).groupby("lab_number").hour.quantile([0.05, 0.5, 0.95]).unstack())

In [ ]:
# Session-end status in the rig settings: SUBJECT_DISENGAGED_TRIGGERED and _TRIALNUM
print("end status (SUBJECT_DISENGAGED_TRIGGERED):", S.disengaged.astype(str).value_counts().to_dict())
x = S[S.disengaged_trial.notna()]
print("sessions with a trigger trial:", len(x), "| trigger trial after the last trial:",
      int((x.disengaged_trial > x.n_trials).sum()))
# Wheel series: data and timestamps should have the same length
wm = S[S.wheel_n.notna() & (S.wheel_n != S.wheel_ts_n)]
print("files without a wheel series:", int(S.wheel_n.isna().sum()), "| wheel data and timestamps of different length:",
      len(wm), "files;", wm.lab_number.value_counts().sort_index().to_dict())
# Clock check: wheel position against the trial table, in 16 sessions drawn at random.
# The wheel displacement from stimulus onset to response should follow the choice when
# the trial times are taken as they are, and not when they are shifted by 30 s.
rng_check = np.random.default_rng(1)
pick = S[(S.n_trials > 300)].sample(16, random_state=1)
res = ib.wheel_many(pick.asset_id.tolist(), [0.0, 30.0, -30.0])
rows = []
for (sid, nick, lab), (disp, ch) in zip(pick[["session_id", "nickname", "lab_number"]].to_numpy(), res):
    if disp is None:
        rows.append(dict(session=sid[:8], mouse=nick, lab=lab, ok=False, reason=ch))
        continue
    m = ch != 0
    r = {f"agree_{int(s):+d}s": np.nanmean(np.sign(disp[s][m]) == -ch[m]) for s in disp}
    rows.append(dict(session=sid[:8], mouse=nick, lab=lab, ok=True, **r))
wheel_check = pd.DataFrame(rows)
print(wheel_check.round(3).to_string())
print("agreement is the fraction of trials whose wheel displacement has the sign of -choice; 0 means the opposite sign")
print("median agreement, true times %.3f; shifted +30 s %.3f; -30 s %.3f"
      % tuple(wheel_check[c].median() for c in ["agree_+0s", "agree_+30s", "agree_-30s"]))

### Selection and proficiency

The paper's mice are those of project `ibl_neuropixel_brainwide_01` (claims file header). The
dataset begins on 2019-06-11, while the paper's mice began training in January 2019 (Methods), so
only mice whose first session in the dataset is a first-stage session (50% and 100% contrasts
only) have their whole training in the dataset; the training-time claims need that. The files
carry no training status, so basic (P1) and full (P2) task proficiency are recomputed from the
trials with the criteria of Appendix 1, table 1d, as the claims file header gives them (answers
of 2026-10-22). P1 follows the pipeline's SessionTrainingStatus: at each session, that session and
the two before it (any protocol); trials completed are all trials, no-response trials included;
percent correct is over contrasts of 50% or more; the 1b criteria are tested and then the 1a
criteria, at any date. The main reading requires "all contrasts introduced" (100, 25, 12.5, 6.25
and 0%) in each of the three sessions. Variants: the pipeline's check of the current session only
(0% and a contrast between 0.1 and 6.5%), and 1b replacing 1a from 2019-09-12. P2 is computed as
before, over three consecutive full-task training days.

In [ ]:
S = S[~S.session_id.isin(dup_ids) & (S.n_trials > 0)].copy()
T = T[T.session_id.isin(S.session_id)].copy()
mice = ib.project_mice(S)
lab = S.groupby("subject_id").lab_number.first()
nick = S.groupby("subject_id").nickname.first()
print("mice in the dataset", S.subject_id.nunique(), "| of the project", len(mice))
print("not in the project:", list(nick[~nick.index.isin(mice)]))
Dall = ib.daily(T, S)
first = Dall.groupby("subject_id").head(1).set_index("subject_id")
first_stage = first.contrasts.apply(lambda c: c == frozenset([50.0, 100.0]))
complete = first_stage[first_stage].index.intersection(mice)
print("project mice whose first session is a first-stage session:", len(complete),
      "| others:", len(mice) - len(complete))
print("first session of the others: task", S[S.subject_id.isin(mice.difference(complete))]
      .sort_values("start").groupby("subject_id").task.first().value_counts().to_dict(),
      "| date", S[S.subject_id.isin(mice.difference(complete))].groupby("subject_id").date.min()
      .value_counts().head(3).to_dict())
print("complete-history mice by lab:", lab[complete].value_counts().sort_index().to_dict())
Sc = S[S.subject_id.isin(complete)]
Tc = T[T.session_id.isin(Sc.session_id)]
Dc = Dall[Dall.subject_id.isin(complete)]
Dfull = ib.daily(Tc, Sc, ["biased"])
Tm = ib.trials_with_days(Tc, Sc, Dc)
print("days with more than one session:", int((Sc.groupby(["subject_id", "date"]).size() > 1).sum()))

In [ ]:
# P1 readings (claims file header, answers of 2026-10-22). Main: a mouse is proficient at the first
# session meeting the 1a or the 1b criteria, at any date, with "all contrasts introduced" in each of
# the three sessions. Variants: the pipeline's contrast check on the current session only ("last"),
# and 1b replacing 1a from 2019-09-12 ("1b-replace").
READINGS = [("either", "each"), ("either", "last"), ("replace", "each"), ("replace", "last")]
MAIN = READINGS[0]
RN = {MAIN: "main", ("either", "last"): "contrasts:last", ("replace", "each"): "1b-replace",
      ("replace", "last"): "1b-replace,contrasts:last"}
SS = ib.session_summary(Tc, Sc)
tday = Dc.set_index(["subject_id", "date"]).tday0

def p1_frame(st):
    st = st[st.p1_session.notna()].copy()
    st["p1_tday0"] = [int(tday[(s, d)]) for s, d in zip(st.index, st.p1_date)]
    return st

STATUS = {r: ib.training_status(SS, Tc, r[1], ib.DATE_1B if r[0] == "replace" else None) for r in READINGS}
P1 = {r: p1_frame(STATUS[r]) for r in READINGS}
P2 = ib.full_proficiency(Dfull, Tc, Sc)
print("sessions of the whole-training mice:", len(SS), "| three-session windows fitted:", len(ib._WINDOW_FITS))
print("sessions with no trial at 50% or more:", int(SS.perf_easy.isna().sum()))
first_biased = Dc[Dc.tasks.apply(lambda t: "biased" in t)].groupby("subject_id").tday0.min()
rows = []
for r in READINGS:
    p = P1[r].p1_tday0
    gap = (first_biased - p).dropna()
    rows.append(dict(reading=RN[r], P1=len(p), **{f"P1 {l}": int((lab[p.index] == l).sum()) for l in ib.LABS},
                     P1_status_1a=int((P1[r].p1_status == "1a").sum()), P1_status_1b=int((P1[r].p1_status == "1b").sum()),
                     reach_1b=int(P1[r].b1_session.notna().sum()),
                     P1_protocol_basic=int(P1[r].p1_session.map(Sc.set_index("session_id").task).eq("training").sum()),
                     first_full_day_minus_P1_median=gap.median(),
                     within_1_to_3_days=f"{int(gap.between(1, 3).sum())}/{len(gap)}",
                     full_task_without_P1=int(len(first_biased.index.difference(p.index)))))
prof = pd.DataFrame(rows).set_index("reading")
print(prof.T.to_string())
print("\nP2 (full task proficiency):", len(P2), "mice;", lab[P2.index].value_counts().sort_index().to_dict())
print("P2 mice without P1 under each reading:", {RN[r]: len(P2.index.difference(P1[r].index)) for r in READINGS})
# Sensitivity that decides nothing: percent correct on 100% contrast only, the wording of Appendix 1, table 1d
alt = p1_frame(ib.training_status(SS, Tc, "each", None, perf="100"))
j = P1[MAIN][["p1_tday0"]].join(alt[["p1_tday0"]], rsuffix="_100", how="outer")
print(f"main reading with percent correct on 100% contrast: {len(alt)} mice reach P1 (against {len(P1[MAIN])}); "
      f"P1 differs for {int((j.p1_tday0 != j.p1_tday0_100).sum())} mice")

## cohort-counts

> Of 206 mice that started training, 140 reached proficiency in the basic task and 98 in the
> full task.

Counts of distinct mice under the paper's selection; proficiency as P1 (four readings) and P2.

In [ ]:
print("project mice in the dataset:", len(mice), "| with their whole training in the dataset:", len(complete))
print("P1 by reading:", {RN[r]: len(P1[r]) for r in READINGS})
print("P2:", len(P2))
exp = [(206, 20.6), (140, 14), (98, 9.8)]
for name, n in [("started training, whole training in the dataset", len(complete)),
                ("project mice with any session", len(mice))]:
    print(f"{name}: {n}, within 206 +- 20.6: {abs(n - 206) <= 20.6}")
for r in READINGS:
    print(f"P1 {RN[r]}: {len(P1[r])}, within 140 +- 14: {abs(len(P1[r]) - 140) <= 14}")
print(f"P2: {len(P2)}, within 98 +- 9.8: {abs(len(P2) - 98) <= 9.8}")

**Result: not reproduced.** The dataset holds 177 mice of the paper's project, and 136 of them
have their whole training in it, against 206 that started training. Under the main reading 83 of
these reach basic task proficiency (85, 70 and 71 under the variants), against 140, and 60 reach
full task proficiency, against 98. No count is within tolerance under any reading. The dataset
holds only sessions from 2019-06-11 on, while training began in January 2019.

## fig2g-days-to-proficiency

> Basic task training took 18.4 +- 13.0 days (s.d., n = 140 mice), from 3 days for the fastest
> mouse to 59 days for the slowest.

Training days up to P1, counting the P1 day (code query branch) or not (CSV branch).

In [ ]:
def check(vals, exp):
    return all(abs(v - e) <= tol for v, (e, tol) in zip(vals, exp))

EXP_DAYS = [(18.4, 1.84), (13.0, 1.3), (3, 0.3), (59, 5.9)]
rows = []
days = {}
for r in READINGS:
    for cnt in ("incl", "excl"):
        d = P1[r].p1_tday0 + (1 if cnt == "incl" else 0)
        days[(r, cnt)] = d
        v = [d.mean(), d.std(), d.min(), d.max()]
        rows.append(dict(reading=RN[r], count=cnt, n=len(d), mean=v[0], sd=v[1], min=v[2], max=v[3],
                         n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_DAYS)),
                         all_in_tol=check(v, EXP_DAYS)))
fig2g = pd.DataFrame(rows)
print(fig2g.round(2).to_string())
d = days[(READINGS[0], "incl")]
print("\nby lab (main, incl):")
print(d.groupby(lab[d.index]).describe()[["count", "mean", "50%", "min", "max"]].round(1))

**Result: partially reproduced.** Under the main reading (83 mice) the mean is 19.87 days counting
the P1 day and 18.87 without, both within 18.4 +- 1.84, and the slowest mouse takes 58 and 57
days, within 59 +- 5.9. The s.d. is 11.39 days, against 13.0 +- 1.3, and the fastest mouse takes 5
or 4 days, against 3 +- 0.3: two of the four values are within tolerance. The variants give means
of 17.73 to 23.17 days, s.d.s of 10.91 to 14.42, and at most two values within tolerance.

## fig2g-days-differ-across-labs

> The number of days to basic task proficiency differed across laboratories; within-lab
> interquartile ranges ranged from 8 days (Lab 2) to 22 days (Lab 6).

In [ ]:
rows = []
for (r, cnt), d in days.items():
    l = lab[d.index]
    iqr = d.groupby(l).quantile(0.75) - d.groupby(l).quantile(0.25)
    p_kw = ib.lab_test(d, l, "kruskal")
    p_code = ib.lab_test(d, l, "code")
    rows.append(dict(reading=RN[r], count=cnt, n=len(d), p_kruskal=p_kw, p_code_switch=p_code,
                     iqr_lab2=iqr.get("Lab 2"), iqr_lab6=iqr.get("Lab 6"),
                     iqr_range=f"{iqr.min():.1f}-{iqr.max():.1f}",
                     p_below_0001=p_kw < 0.001,
                     iqr_in_tol=abs(iqr.get("Lab 2", np.nan) - 8) <= 0.8 and abs(iqr.get("Lab 6", np.nan) - 22) <= 2.2))
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))
print("mice per lab (main):", lab[days[(READINGS[0], 'incl')].index].value_counts().sort_index().to_dict())

**Result: not reproduced.** Under the main reading the Kruskal-Wallis p across labs is 0.029,
above the paper's p < 0.001 (0.023 to 0.088 under the variants). The Lab 2 interquartile range is
7.75 days, within 8 +- 0.8, and the Lab 6 range 5.75 days, against 22 +- 2.2; the within-lab
ranges run from 5.8 to 12.5 days.

## fig2s1-trials-to-proficiency

> Measured in trials, basic task training took 10.8 +- 8.6 thousand trials (s.d., n = 140), from
> one thousand for the fastest mouse to 43 thousand for the slowest; reaching 80% correct on easy
> trials took on average about 7.4 thousand trials.

In [ ]:
EXP_TR = [(10800, 1080), (8600, 860), (1000, 100), (43000, 4300)]
rows = []
for (r, cnt), d in days.items():
    dd = Dc[Dc.subject_id.isin(d.index)]
    lim = dd.subject_id.map(d)
    # days counted: tday0 < days (excl) or <= p1 day (incl); d is the count, so tday0 < d
    tr = dd[dd.tday0 < lim].groupby("subject_id").n_trials.sum()
    v = [tr.mean(), tr.std(), tr.min(), tr.max()]
    rows.append(dict(reading=RN[r], count=cnt, n=len(tr), mean=v[0], sd=v[1], min=v[2], max=v[3],
                     n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_TR)), all_in_tol=check(v, EXP_TR)))
print(pd.DataFrame(rows).round(0).to_string())

def trials_to_80(mice_idx, variant=False):
    dd = Dc[Dc.subject_id.isin(mice_idx) & (Dc.n_trials > 100)].sort_values(["subject_id", "tday0"]).copy()
    dd["perf_f"] = dd.groupby("subject_id").perf_easy.transform(
        lambda x: x.rolling(3, center=True, min_periods=1).median())
    dd["day"] = dd.groupby("subject_id").cumcount()
    dd["cum"] = dd.groupby("subject_id").n_trials.cumsum()
    if variant:
        f = dd[dd.perf_f > 80].groupby("subject_id").cum.first()
        return f.mean(), len(f)
    g = dd.groupby("day").agg(perf=("perf_f", "mean"), cum=("cum", "mean"), n=("subject_id", "size"))
    hit = g[g.perf > 80]
    return (hit.cum.min() if len(hit) else np.nan), int(hit.index.min()) if len(hit) else None

for r in READINGS:
    v, day = trials_to_80(P1[r].index)
    vv, nn = trials_to_80(P1[r].index, variant=True)
    print(f"{RN[r]}: trials to a mean of 80% on easy trials {v:.0f} (first at training day {day}); "
          f"within 7400 +- 740: {abs(v - 7400) <= 740} | variant, mean of each mouse's first: {vv:.0f} (n={nn})")

**Result: partially reproduced.** Under the main reading the mean number of trials to P1 is 11,839
counting the P1 day and 11,104 without, within 10,800 +- 1,080. The s.d. (7,515 and 7,472), the
fastest mouse (2,357 and 1,687 trials) and the slowest (34,998 and 34,265) are outside tolerance.
The trial count at which the mean easy-trial performance first exceeds 80% is 7,329, within 7,400
+- 740 (6,659 to 7,306 under the variants).

## fig2h-proportion-learned-40-days

> About 80% of mice that started training would learn the basic task within the 40 days usually
> allotted (Kaplan-Meier estimate over all 206 mice).

In [ ]:
from lifelines import KaplanMeierFitter
total_days = Dc.groupby("subject_id").tday0.max() + 1
last_date = Dc.groupby("subject_id").date.max()
rows = []
for r in READINGS:
    for cnt in ("incl", "excl"):
        for variant in ("main", "drop_in_training"):
            idx = complete
            p = days[(r, cnt)]
            dur = pd.Series(np.where(idx.isin(p.index), p.reindex(idx), total_days.reindex(idx)), index=idx)
            ev = pd.Series(idx.isin(p.index), index=idx)
            if variant == "drop_in_training":
                # still in training at the cutoff: no P1 and a session in the last 7 days before it
                still = (~ev) & (last_date.reindex(idx) >= "2020-03-16")
                dur, ev = dur[~still], ev[~still]
            km = KaplanMeierFitter().fit(dur, ev)
            at40 = 1 - km.predict(40)
            atlast = 1 - km.predict(dur.max())
            rows.append(dict(reading=RN[r], count=cnt, variant=variant, n=len(dur), events=int(ev.sum()),
                             p40=at40, p_last=atlast, last_day=dur.max(),
                             ok=abs(at40 - 0.8) <= 0.08 and atlast > at40))
print(pd.DataFrame(rows).round(3).to_string())

**Result: reproduced.** Under the main reading ("all contrasts introduced" in each of the three
sessions), over the 136 mice that started training, mice without P1 censored, the Kaplan-Meier
proportion at day 40 is 0.749 counting the P1 day and 0.741 without, within 0.8 +- 0.08, and 1.000
at the last day. The pipeline's contrast check, a variant, gives 0.768 and 0.760. With 1b
replacing 1a from 2019-09-12 the proportion is 0.591 to 0.602, outside tolerance. The code's
variant, leaving out mice still in training at the cutoff, gives 0.815 and 0.809 under the main
reading.

## fig2bc-threshold-bias-iqr

> Contrast threshold and bias varied little during training; the threshold interquartile range
> slightly decreased from 12-29% in the first 10 training days to 11-23% thereafter, and the bias
> interquartile range went from -5.2 to 7.1% to -5.6 to 6.5%.

Per training day (all sessions of the day pooled), 50:50 days with more than 100 trials and six
or more signed contrasts are fitted. Training day counted from 0 at the mouse's first session.
50:50 sessions are the basic-task sessions with all their trials, repeat trials included (claims
file header, answer of 2026-10-22), so 50:50 days are days with basic-task sessions only.

In [ ]:
keep = Dc[(Dc.n_trials > 100) & Dc.basic_only & (Dc.n_signed >= 6)]
tk = Tm.merge(keep[["subject_id", "date"]], on=["subject_id", "date"])
day_fits = ib.mouse_fits(tk, by=("subject_id", "tday0")).reset_index()
day_fits = day_fits.sort_values(["subject_id", "tday0"])
for v in ("threshold", "bias"):
    day_fits[v + "_f"] = day_fits.groupby("subject_id")[v].transform(
        lambda x: x.rolling(3, center=True, min_periods=1).median())
print("day fits:", len(day_fits), "of", len(keep), "days;", day_fits.threshold.isna().sum(), "without a fit")
print("threshold range %.1f-%.1f, bias range %.1f-%.1f" % (day_fits.threshold.min(), day_fits.threshold.max(),
                                                            day_fits.bias.min(), day_fits.bias.max()))
EXP_BC = [(12, 1.2), (29, 2.9), (11, 1.1), (23, 2.3), (-5.2, 0.52), (7.1, 0.71), (-5.6, 0.56), (6.5, 0.65)]
rows = []
for r in READINGS:
    f = day_fits[day_fits.subject_id.isin(P1[r].index)]
    for filt in ("filtered", "unfiltered"):
        sfx = "_f" if filt == "filtered" else ""
        early = f[f.tday0 <= 9]
        for after in ("10-39", "10+"):
            late = f[(f.tday0 >= 10) & ((f.tday0 <= 39) if after == "10-39" else True)]
            v = [early["threshold" + sfx].quantile(.25), early["threshold" + sfx].quantile(.75),
                 late["threshold" + sfx].quantile(.25), late["threshold" + sfx].quantile(.75),
                 early["bias" + sfx].quantile(.25), early["bias" + sfx].quantile(.75),
                 late["bias" + sfx].quantile(.25), late["bias" + sfx].quantile(.75)]
            narrower = (v[3] - v[2]) < (v[1] - v[0])
            rows.append(dict(reading=RN[r], filt=filt, after=after, n_early=len(early), n_late=len(late),
                             thr_early=f"{v[0]:.1f}-{v[1]:.1f}", thr_late=f"{v[2]:.1f}-{v[3]:.1f}",
                             bias_early=f"{v[4]:.1f}-{v[5]:.1f}", bias_late=f"{v[6]:.1f}-{v[7]:.1f}",
                             n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_BC)),
                             narrower=narrower, ok=check(v, EXP_BC) and narrower))
fig2bc = pd.DataFrame(rows)
print(fig2bc.to_string())

**Result: partially reproduced.** Under every reading the threshold interquartile range after day
10 is narrower than in the first 10 days (main, filtered, days 10 to 39: 11.5-23.7% against
11.1-24.3%). No reading puts all eight bounds within tolerance; the main reading puts at most four
(filtered, days 10 on: threshold 11.1-24.3% then 11.5-23.0%, bias -8.4 to 6.1% then -8.6 to 7.1%).
The early threshold range is narrower than the paper's 12-29%, and the bias ranges are shifted to
the left.

## fig2-abs-bias-threshold-during-training

> During training mice were not systematically biased to one side; absolute bias averaged
> 9.8 +- 12.1 (s.d.), and contrast threshold, which stabilized after about 10 sessions, averaged
> 17.8 +- 11.7%.

In [ ]:
EXP_AB = [(9.8, 0.98), (12.1, 1.21), (17.8, 1.78), (11.7, 1.17)]
rows = []
for r in READINGS:
    f = day_fits[day_fits.subject_id.isin(P1[r].index)].copy()
    f["lab"] = lab[f.subject_id].to_numpy()
    for filt in ("filtered", "unfiltered"):
        sfx = "_f" if filt == "filtered" else ""
        ab = f["bias" + sfx].abs()
        for after in ("10-39", "10+"):
            late = f[(f.tday0 >= 10) & ((f.tday0 <= 39) if after == "10-39" else True)]
            th = late["threshold" + sfx]
            for avg in ("pooled", "lab means"):
                if avg == "pooled":
                    m_ab, m_th = ab.mean(), th.mean()
                else:
                    m_ab = ab.groupby(f.lab).mean().mean()
                    m_th = th.groupby(late.lab).mean().mean()
                v = [m_ab, ab.std(), m_th, th.std()]
                rows.append(dict(reading=RN[r], filt=filt, after=after, avg=avg, abs_bias=v[0], abs_bias_sd=v[1],
                                 thr=v[2], thr_sd=v[3],
                                 means_ok=abs(v[0] - 9.8) <= 0.98 and abs(v[2] - 17.8) <= 1.78,
                                 sds_ok=abs(v[1] - 12.1) <= 1.21 and abs(v[3] - 11.7) <= 1.17))
print(pd.DataFrame(rows).round(3).to_string())
f = day_fits[day_fits.subject_id.isin(P1[READINGS[0]].index)]
print("\nsigned bias over all days (main, filtered): mean %.2f, median %.2f; per lab mean:"
      % (f.bias_f.mean(), f.bias_f.median()),
      f.bias_f.groupby(lab[f.subject_id].to_numpy()).mean().round(2).to_dict())

**Result: reproduced.** Under the main reading, without the median filter, the mean absolute bias
over all days is 9.759 and the mean threshold after day 10 is 18.56% (days 10 to 39) and 18.30%
(days 10 on), within tolerance of 9.8 and 17.8; with the filter the absolute bias is 8.821, at the
lower edge of 9.8 +- 0.98. The s.d.s are smaller than the paper's under every reading (absolute
bias 6.30 to 8.72 against 12.1; threshold 8.55 to 9.89 against 11.7). The mean signed bias is
-0.99 (main, filtered), with lab means from -5.11 to 2.55; the paper's "similar across
laboratories" is not tested by the criterion.

## fig2s2-variability-decreases

> Variability in performance across mice decreased with training but did not disappear; between
> training days 15 and 40, mean easy-trial performance rose from 80.7% to 91.1% while its s.d.
> across mice fell from 19.1% to 10.1%.

In [ ]:
EXP_V = [(80.7, 8.07), (91.1, 9.11), (19.1, 1.91), (10.1, 1.01)]

def perf_by_day(mice_idx, only_5050):
    dd = Dc[Dc.subject_id.isin(mice_idx) & (Dc.n_trials > 100)]
    if only_5050:
        dd = dd[dd.basic_only]
    dd = dd.sort_values(["subject_id", "tday0"]).copy()
    dd["perf_f"] = dd.groupby("subject_id").perf_easy.transform(
        lambda x: x.rolling(3, center=True, min_periods=1).median())
    dd["day"] = dd.groupby("subject_id").cumcount()
    return dd.groupby("day").perf_f.agg(["mean", "std", "size"])

rows = []
for r in READINGS:
    idx = P1[r].index
    for name, g in (("(1) all sessions", perf_by_day(idx, False)), ("(2) basic-task days only", perf_by_day(idx, True))):
        for a, b in ((15, 40), (14, 39)):
            if b not in g.index:
                continue
            v = [g.loc[a, "mean"], g.loc[b, "mean"], g.loc[a, "std"], g.loc[b, "std"]]
            rows.append(dict(reading=RN[r], data=name, days=f"{a},{b}", n=f"{g.loc[a, 'size']},{g.loc[b, 'size']}",
                             mean_a=v[0], mean_b=v[1], sd_a=v[2], sd_b=v[3],
                             n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_V)),
                             ok=check(v, EXP_V) and v[1] > v[0] and v[3] < v[2]))
    # (3) unfiltered per-day performance, days strictly within 2 of the bin centre; mean per mouse in the bin
    dd = Dc[Dc.subject_id.isin(idx)]
    vals = []
    for c in (15, 39):
        b = dd[(dd.tday0 - c).abs() < 2].groupby("subject_id").perf_easy.mean()
        vals.append(b)
    v = [vals[0].mean(), vals[1].mean(), vals[0].std(), vals[1].std()]
    rows.append(dict(reading=RN[r], data="(3) binned", days="15,39", n=f"{len(vals[0])},{len(vals[1])}",
                     mean_a=v[0], mean_b=v[1], sd_a=v[2], sd_b=v[3],
                     n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_V)),
                     ok=check(v, EXP_V) and v[1] > v[0] and v[3] < v[2]))
print(pd.DataFrame(rows).round(2).to_string())

**Result: partially reproduced.** Under every reading the mean easy-trial performance rises and its
s.d. across mice falls between days 15 and 40 (main, all sessions: 86.13 to 91.11%, s.d. 15.17 to
7.98%). The two means are within tolerance under most readings, but no reading has both s.d.s
within tolerance (15.03 to 17.12% at day 15 against 19.1%; 6.86 to 12.19% at day 40 against
10.1%).

## early-sessions-predict-slow-learners

> A Random Forests decoder applied to the change in easy-trial performance over the first five
> sessions predicted whether a mouse would be in the bottom quartile of learning speed with 53%
> accuracy (chance 25%), and misclassified a top-quartile mouse as bottom-quartile only 7% of the
> time.

The readings of the criterion: features (a) four differences, (b) five performances; labels
(A) four quartiles, (B) bottom quartile against the rest; quartiles from days to P1 or trials to
P1; accuracy (i) recall of the bottom quartile, (ii) four-class accuracy under (A), (iii)
precision of the bottom quartile. scikit-learn RandomForestClassifier with default settings,
leave-one-out; its seed is a standard choice (0).
The criterion's readings are run under the main reading of P1 only, to keep the notebook within
its time limit.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
first5 = Dc[Dc.tday0 <= 4].pivot(index="subject_id", columns="tday0", values="perf_easy")
rows = []
for r in [MAIN]:
    for qsrc in ("days", "trials"):
        d = days[(r, "incl")]
        if qsrc == "trials":
            dd = Dc[Dc.subject_id.isin(d.index)]
            d = dd[dd.tday0 < dd.subject_id.map(d)].groupby("subject_id").n_trials.sum()
        idx = d.index.intersection(first5.dropna().index)
        q = pd.qcut(d[idx].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)  # 4 = slowest
        F = {"a": np.diff(first5.loc[idx].to_numpy(), axis=1), "b": first5.loc[idx].to_numpy()}
        for feat, X in F.items():
            for lab_kind in ("A", "B"):
                y = q.to_numpy() if lab_kind == "A" else (q.to_numpy() == 4).astype(int)
                pred = cross_val_predict(RandomForestClassifier(random_state=0, n_jobs=4), X, y, cv=LeaveOneOut())
                bottom = 4 if lab_kind == "A" else 1
                recall = np.mean(pred[y == bottom] == bottom)
                precision = np.mean(y[pred == bottom] == bottom) if (pred == bottom).any() else np.nan
                acc4 = np.mean(pred == y) if lab_kind == "A" else np.nan
                top_as_bottom = np.mean(pred[q.to_numpy() == 1] == bottom)
                rows.append(dict(reading=RN[r], quartiles=qsrc, features=feat, labels=lab_kind, n=len(idx),
                                 recall=recall, four_class=acc4, precision=precision, top_as_bottom=top_as_bottom,
                                 base_rate=np.mean(y == bottom)))
es = pd.DataFrame(rows)
print(es.round(3).to_string())
acc = pd.concat([es.recall, es.four_class.dropna(), es.precision])
print("\naccuracy readings within 0.53 +- 0.053: %d of %d; range %.3f-%.3f"
      % (int(((acc - 0.53).abs() <= 0.053).sum()), len(acc), acc.min(), acc.max()))
print("top quartile classed as bottom, range %.3f-%.3f; within 0.07 +- 0.007: %d of %d"
      % (es.top_as_bottom.min(), es.top_as_bottom.max(), int(((es.top_as_bottom - 0.07).abs() <= 0.007).sum()), len(es)))

**Result: blocked (methods_underspecified).** Under the main reading of P1 the criterion's readings
disagree on whether the accuracy is near 53%: 3 of 20 accuracy values are within 0.53 +- 0.053
(features (b), four quartiles from days: recall 0.571, four-class accuracy 0.518, precision
0.571), and the others run from 0.143 to 0.667. The share of top-quartile mice classed as bottom
quartile ranges from 0.000 to 0.143 (within 0.07 +- 0.007 in none of 8). The repository holds no
code for this analysis.

## Sessions at basic task proficiency (S1)

The basic (training protocol) sessions of the P1 day and the two training days before it, per
reading of P1. One value per mouse from the pooled trials.

In [ ]:
def s1_values(r):
    t = ib.window(Tm, P1[r].p1_tday0, -2, 0, ["training"])
    fits = ib.mouse_fits(t)
    g = t.groupby("subject_id")
    ses = Sc[Sc.session_id.isin(t.session_id)]
    v = pd.DataFrame({
        "perf_easy": g.apply(ib.perf_easy, include_groups=False),
        "rt": g.rt.median(),
        "n_trials": ses.groupby("subject_id").n_trials.mean(),
        "threshold": fits.threshold, "bias": fits.bias,
        "trials_per_min": (ses.n_trials / ses.duration_min).groupby(ses.subject_id).mean(),
    })
    v["lab"] = lab[v.index]
    return v, t

S1 = {}
S1_trials = {}
for r in READINGS:
    S1[r], S1_trials[r] = s1_values(r)
    print(RN[r], "mice", len(S1[r]), "| sessions per mouse", S1_trials[r].groupby("subject_id").session_id.nunique()
          .value_counts().to_dict(), "| no valid trial duration", int(S1[r].rt.isna().sum()))

## fig3c-easy-errors-basic

> At basic task proficiency, errors on easy (50% and 100%) contrasts were low, 9.5 +- 3.6%, with
> no significant difference across laboratories.

In [ ]:
rows = []
for r in READINGS:
    v = S1[r][S1[r].rt.notna()]
    e = 100 - v.perf_easy
    rows.append(dict(reading=RN[r], n=len(e), mean=e.mean(), sd=e.std(), min=e.min(), max=e.max(),
                     mean_ok=abs(e.mean() - 9.5) <= 0.95, sd_ok=abs(e.std() - 3.6) <= 0.36))
print(pd.DataFrame(rows).round(2).to_string())
v = S1[READINGS[0]]
print("by lab (main):", (100 - v.perf_easy).groupby(v.lab).mean().round(2).to_dict())

**Result: partially reproduced.** Under the main reading the error rate on easy trials at
proficiency is 8.41 +- 3.79% (81 mice): the s.d. is within tolerance of 3.6 +- 0.36, the mean just
outside 9.5 +- 0.95. The pipeline's contrast check, a variant, gives 9.66 +- 3.69% (85 mice), both
within tolerance; with 1b replacing 1a it is 5.26 and 5.84%, as the 1b criteria require more than
90% correct. Two of the 83 mice with P1 have no basic-task session in S1. The lab comparison is
the next claim.

## fig3de-threshold-bias-basic

> At basic task proficiency the contrast threshold was similar across laboratories at
> 14.3 +- 3.8 (s.d., n = 7 laboratories) and the response bias was small at 0.3 +- 8.4.

In [ ]:
EXP_DE = [(14.3, 1.43), (0.3, 3.2), (3.8, 0.38), (8.4, 0.84)]
rows = []
for r in READINGS:
    v = S1[r]
    for how in ("over mice", "over lab means"):
        if how == "over mice":
            th, bi = v.threshold.dropna(), v.bias.dropna()
        else:
            th, bi = v.threshold.groupby(v.lab).mean(), v.bias.groupby(v.lab).mean()
        x = [th.mean(), bi.mean(), th.std(), bi.std()]
        rows.append(dict(reading=RN[r], how=how, n=len(th), thr=x[0], bias=x[1], thr_sd=x[2], bias_sd=x[3],
                         n_in_tol=sum(abs(a - e) <= t for a, (e, t) in zip(x, EXP_DE)),
                         means_ok=abs(x[0] - 14.3) <= 1.43 and abs(x[1] - 0.3) <= 3.2, all_ok=check(x, EXP_DE)))
print(pd.DataFrame(rows).round(2).to_string())
print("fits without a value:", {RN[r]: int(S1[r].threshold.isna().sum()) for r in READINGS})

**Result: reproduced.** Over mice, under the main reading, the threshold is 14.09 +- 3.80 and the
bias -0.46 +- 8.09 (81 mice): all four values within tolerance, and also under the pipeline's
contrast check (13.89 +- 3.92 and 0.16 +- 9.05). Over the seven lab means the means match (13.81
to 14.55) but the s.d.s are 1.14 to 2.24, far below 3.8 and 8.4, so the paper's s.d.s are over
mice.

## fig3cde-no-lab-difference-medians and fig3cde-no-lab-difference-variances

> At basic task proficiency, easy-trial performance, contrast threshold and bias did not differ
> in median across laboratories.

> At basic task proficiency, easy-trial performance, contrast threshold and bias did not differ
> in variance across laboratories.

Kruskal-Wallis (and Levene) across labs on six variables, Benjamini-Hochberg over the six.

In [ ]:
VARS = ["perf_easy", "rt", "n_trials", "threshold", "bias", "trials_per_min"]
labtests = {}
for r in READINGS:
    v = S1[r][S1[r].rt.notna()]
    out = {}
    for test in ("kruskal", "levene", "code"):
        p = np.array([ib.lab_test(v[x], v.lab, test) for x in VARS])
        out[test] = pd.Series(p, index=VARS)
        out[test + "_fdr"] = pd.Series(ib.fdr(p), index=VARS)
    labtests[r] = pd.DataFrame(out)
    print(RN[r], "n =", len(v), "| mice per lab", v.lab.value_counts().sort_index().to_dict())
    print(labtests[r].round(4).to_string(), "\n")

In [ ]:
for r in READINGS:
    lt = labtests[r]
    m = lt.loc[["perf_easy", "threshold", "bias"]]
    print(f"{RN[r]}: medians, all three FDR p >= 0.05: {bool((m.kruskal_fdr >= 0.05).all())} "
          f"(min {m.kruskal_fdr.min():.3f}); code switch: {bool((m.code_fdr >= 0.05).all())} | "
          f"variances, all three FDR p >= 0.05: {bool((m.levene_fdr >= 0.05).all())} (min {m.levene_fdr.min():.3f})")
    v = S1[r]
    eff = {x: f"{v[x].groupby(v.lab).median().min():.2f}..{v[x].groupby(v.lab).median().max():.2f}"
           for x in ["perf_easy", "threshold", "bias"]}
    sds = {x: f"{v[x].groupby(v.lab).std().min():.2f}..{v[x].groupby(v.lab).std().max():.2f}"
           for x in ["perf_easy", "threshold", "bias"]}
    print("   lab medians:", eff, "| lab s.d.:", sds)

**Result: reproduced.** Under the main reading the corrected Kruskal-Wallis p values are 0.287 for
performance, 0.439 for threshold and 0.730 for bias, all above 0.05 (lab medians: performance 89.0
to 93.1%, threshold 12.3 to 17.1, bias -6.1 to 1.8); the paper's are 0.63, 0.81 and 0.81. The
code's switch to ANOVA gives the same. The pipeline's contrast check gives the same outcome
(smallest 0.395); with 1b replacing 1a, a variant, performance differs across labs (corrected p =
0.038 and 0.026).

**Result: reproduced.** The corrected Levene p values for performance, threshold and bias are all
above 0.05 under the main reading (0.268, 0.268 and 0.299) and under every variant (smallest
0.212).

## fig3s2a-trial-duration

> At basic task proficiency the median trial duration (stimulus onset to feedback) was
> 468 +- 221 ms, with some differences across laboratories.

In [ ]:
rows = []
for r in READINGS:
    v = S1[r][S1[r].rt.notna()]
    rt = 1000 * v.rt
    pk = labtests[r].loc["rt", "kruskal"]
    pf = labtests[r].loc["rt", "kruskal_fdr"]
    rows.append(dict(reading=RN[r], n=len(rt), median=rt.median(), sd=rt.std(), mean=rt.mean(),
                     p_kw=pk, p_kw_fdr=pf,
                     ok_uncorr=abs(rt.median() - 468) <= 46.8 and abs(rt.std() - 221) <= 22.1 and pk < 0.05,
                     ok_fdr=abs(rt.median() - 468) <= 46.8 and abs(rt.std() - 221) <= 22.1 and pf < 0.05))
print(pd.DataFrame(rows).round(4).to_string())
v = S1[READINGS[0]]
print("lab medians (ms, main):", (1000 * v.rt).groupby(v.lab).median().round(0).to_dict())

**Result: not reproduced.** Under the main reading the median over mice of the median trial
duration is 420.5 ms, just below 468 +- 46.8, with s.d. 219 ms (within 221 +- 22.1), and the
Kruskal-Wallis test does not find a difference across labs (p = 0.061 uncorrected, 0.122
corrected), which the criterion requires (p < 0.05). The pipeline's contrast check gives 438.4
+- 225.7 ms, within tolerance, with p = 0.068. Lab 4 is slower than the others (median 702 ms
against 370 to 455 ms).

## fig3s2b-trials-per-session

> At basic task proficiency mice performed 719 +- 223 trials per session; the difference across
> laboratories was significant but only in one laboratory relative to the rest.

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
rows = []
for r in READINGS:
    v = S1[r][S1[r].rt.notna()]
    p = ib.lab_test(v.n_trials, v.lab, "anova")
    tk = pairwise_tukeyhsd(v.n_trials, v.lab)
    tab = pd.DataFrame(tk._results_table.data[1:], columns=tk._results_table.data[0])
    sig = tab[tab.reject]
    labs_inv = pd.concat([sig.group1, sig.group2]).value_counts()
    common = [l for l in labs_inv.index if all((sig.group1 == l) | (sig.group2 == l))] if len(sig) else []
    rows.append(dict(reading=RN[r], n=len(v), mean=v.n_trials.mean(), sd=v.n_trials.std(), p_anova=p,
                     p_anova_fdr=labtests[r].loc["n_trials", "kruskal_fdr"],
                     sig_pairs=", ".join(f"{a}-{b}" for a, b in zip(sig.group1, sig.group2)),
                     one_lab=common[0] if common else "",
                     values_ok=abs(v.n_trials.mean() - 719) <= 71.9 and abs(v.n_trials.std() - 223) <= 22.3))
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))
v = S1[READINGS[0]]
print("lab means (main):", v.n_trials.groupby(v.lab).mean().round(0).to_dict())

**Result: partially reproduced.** Under the main reading mice performed 706.8 +- 196.9 trials per
session (81 mice): the mean is within 719 +- 71.9, the s.d. below 223 +- 22.3. The one-way ANOVA
across labs gives p = 1.2e-04, below 0.05 but not below 1e-6, and the five significant Tukey pairs
all involve Lab 7 (412 trials per session against 635 to 836). The variants give the same pattern
(p = 9.2e-05 to 0.0026, every pair with Lab 7).

## lab-link

The lab numbers (Lab 1 to 7) and, for the positive control, the labs' time zones come from the
paper's code by institution (claims file header). The dataset names each session's institution.

In [ ]:
inst = S.groupby("institution").agg(lab_number=("lab_number", "first"), mice=("subject_id", "nunique"),
                                    labs_in_files=("lab", lambda x: sorted(set(x))))
inst["time_zone"] = inst.lab_number.map(ib.TIME_ZONE)
print(inst.to_string())
print("institutions matched to a lab number:", int(inst.lab_number.notna().sum()), "of", len(inst),
      "| distinct lab numbers:", inst.lab_number.nunique())
h = pd.to_datetime(S.start.str[:19]).dt.hour + pd.to_datetime(S.start.str[:19]).dt.minute / 60
med = h.groupby(S.lab_number).median()
print("median start hour by lab (stated as UTC):", med.round(1).to_dict())
print("spread of lab medians %.1f h; if the times were UTC, Lab 7 (UTC-7) would start about 7 h later than Lab 1"
      % (med.max() - med.min()))

**Result: reproduced.** The seven institutions in the files match the code's seven labs one to
one (7 of 7), with CSHL holding two lab groups (churchlandlab and zadorlab) and SWC two (hoferlab
and mrsicflogellab). The time zones cannot be confirmed from the data: `session_start_time` is
stated in UTC, but the median start hours of all labs fall between 11.0 and 14.4 h, Lab 7 at 11.0
h like Lab 1, so the times are local clock times labelled UTC.

## fig3s1a-choice-consistency-basic

> In the basic task, mouse choices were no more consistent within laboratories than across
> laboratories.

Fraction of rightward choices per signed contrast (50% contrast trials dropped), per mouse; 10,000
shuffles of lab labels. Main: S1 sessions; variant (code): the P1 day and the two days after.

In [ ]:
def choice_matrix(t, by_block=False):
    t = t[(t.abs_contrast != 50) & t.choice_right.notna()]
    cols = ["signed_contrast"] + (["probabilityLeft"] if by_block else [])
    return t.groupby(["subject_id"] + cols).choice_right.mean().unstack(cols)

rows = []
for r in READINGS:
    for win in ("S1", "code [0, 2]"):
        t = S1_trials[r] if win == "S1" else ib.window(Tm, P1[r].p1_tday0, 0, 2, ["training", "biased"])
        if win != "S1":  # basic-task trials, and the 50:50 trials of full-task sessions
            t = t[(t.task == "training") | np.isclose(t.probabilityLeft, 0.5)]
        M = choice_matrix(t)
        obs, null, p = ib.consistency_test(M, lab[M.index], 10000, seed=11)
        rows.append(dict(reading=RN[r], sessions=win, mice=len(M), consistency=obs,
                         null_mean=null.mean(), p=p, ok=p >= 0.05))
print(pd.DataFrame(rows).round(4).to_string())

**Result: reproduced.** Choices are not more consistent within labs than in shuffles: p = 0.458
on S1 and 0.535 on the code's sessions under the main reading, and 0.440 to 0.861 under the
variants.

## Sessions at full task proficiency (S2)

The full task (biased protocol) sessions of the P2 day and the two training days before it.

In [ ]:
S2t = ib.window(Tm, P2.p2_tday0, -2, 0, ["biased"])
print("P2 mice", len(P2), "| sessions per mouse", S2t.groupby("subject_id").session_id.nunique().value_counts().to_dict(),
      "| trials", len(S2t), "| block values", S2t.probabilityLeft.round(2).value_counts().to_dict())
blk = {"2080": 0.2, "8020": 0.8}
S2fits = ib.mouse_fits(S2t.assign(block=np.select([np.isclose(S2t.probabilityLeft, 0.2),
                                                    np.isclose(S2t.probabilityLeft, 0.8)],
                                                   ["2080", "8020"], "5050")),
                       by=("subject_id", "block")).unstack("block")
print("fits without a value:", int(S2fits.isna().any(axis=1).sum()))

## fig3s1bc-choice-consistency-full

> In the full task, neither choice fractions nor the bias shift between blocks were more
> consistent within laboratories than expected by chance.

In [ ]:
Mb = choice_matrix(S2t, by_block=True)
obs_b, null_b, p_b = ib.consistency_test(Mb, lab[Mb.index], 10000, seed=12)
Mfrac = choice_matrix(S2t, by_block=True)
shift = Mfrac.xs(0.2, axis=1, level="probabilityLeft") - Mfrac.xs(0.8, axis=1, level="probabilityLeft")
obs_c, null_c, p_c = ib.consistency_test(shift, lab[shift.index], 10000, seed=13)
print(f"(b) choices per contrast and block: mice {len(Mb)}, consistency {obs_b:.3f}, null mean {null_b.mean():.3f}, p = {p_b:.4f}")
print(f"(c) bias shift per contrast: mice {len(shift)}, consistency {obs_c:.3f}, null mean {null_c.mean():.3f}, p = {p_c:.4f}")
print("both p >= 0.05:", p_b >= 0.05 and p_c >= 0.05)

**Result: not reproduced.** In the full task, choices per contrast and block are more consistent
within labs than in shuffles (consistency 98.2 against a null mean of 82.5, p = 0.017, 60 mice).
The bias shift is not (p = 0.154), as in the paper.

## fig3s4-looser-criterion

> With a looser proficiency criterion (80% correct on easy trials over three sessions, n = 150
> mice), behavior was again consistent across laboratories; only the number of trials per session
> differed slightly, and a classifier failed to identify a mouse's laboratory.

For each mouse, basic-task sessions with five or more signed contrasts; the first three
consecutive such sessions each above 80% correct on easy trials.

In [ ]:
ses_tr = Sc[Sc.task == "training"].sort_values("start")
st = Tm[Tm.task == "training"].groupby("session_id").agg(perf=("correct", lambda x: np.nan), n_signed=("signed_contrast", "nunique"))
st["perf"] = Tm[Tm.task == "training"].groupby("session_id").apply(ib.perf_easy, include_groups=False)
ses_tr = ses_tr.join(st, on="session_id")
ses_tr = ses_tr[ses_tr.n_signed >= 5]
loose = {}
for sid, d in ses_tr.groupby("subject_id"):
    ok = (d.perf > 80).to_numpy()
    for j in range(2, len(d)):
        if ok[j - 2: j + 1].all():
            loose[sid] = d.session_id.iloc[j - 2: j + 1].tolist()
            break
print("mice meeting the looser criterion:", len(loose), "| within 150 +- 15:", abs(len(loose) - 150) <= 15)
print("by lab:", lab[list(loose)].value_counts().sort_index().to_dict())
tl = Tm[Tm.session_id.isin({x for v in loose.values() for x in v})]
fl = ib.mouse_fits(tl)
g = tl.groupby("subject_id")
sl = Sc[Sc.session_id.isin(tl.session_id)]
vl = pd.DataFrame({"perf_easy": g.apply(ib.perf_easy, include_groups=False), "rt": g.rt.median(),
                   "n_trials": sl.groupby("subject_id").n_trials.mean(), "threshold": fl.threshold, "bias": fl.bias,
                   "trials_per_min": (sl.n_trials / sl.duration_min).groupby(sl.subject_id).mean()})
vl["lab"] = lab[vl.index]
p = np.array([ib.lab_test(vl[x], vl.lab, "kruskal") for x in VARS])
pc = np.array([ib.lab_test(vl[x], vl.lab, "code") for x in VARS])
print(pd.DataFrame({"kruskal": p, "kruskal_fdr": ib.fdr(p), "code_switch_fdr": ib.fdr(pc)}, index=VARS).round(4))
pf = pd.Series(ib.fdr(p), index=VARS)
print("trials per session corrected p < 0.05:", pf["n_trials"] < 0.05,
      "| other five >= 0.05:", bool((pf.drop("n_trials") >= 0.05).all()))

The classifier part of this claim runs in the section on the lab classifiers below, which
gives its conclusion.

## First full-task sessions (P1b)

Claims file header, answer of 2026-10-22. Main reading: P1b is the first full-task (biased
protocol) session starting at or after the P1 session. Variant, the code's: the first full-task
session at or after the first session meeting the 1b criteria (the pipeline gives a biased session
status trained_1b when the mouse had it before or its last three sessions meet the 1b criteria); a
mouse that never meets them has no P1b. The variant is run under the main reading of P1.

In [ ]:
def p1b_day(r, kind):
    """Training day of the first full-task (biased protocol) session starting at or after P1
    (legend, main) or at or after the first session meeting the 1b criteria (code, variant)."""
    p = P1[r]
    ref = p.p1_session if kind == "legend" else p.b1_session.dropna()
    start = Sc.set_index("session_id").start
    s0 = pd.Series(start.reindex(ref.to_numpy()).to_numpy(), index=ref.index)
    ses = Sc[(Sc.task == "biased") & Sc.subject_id.isin(s0.index)]
    ses = ses[ses.start >= ses.subject_id.map(s0)].sort_values("start").groupby("subject_id").first()
    return pd.Series([int(tday[(s, d)]) for s, d in zip(ses.index, ses.date)], index=ses.index, dtype=int)

PB = [(MAIN, "legend"), (MAIN, "code")] + [(r, "legend") for r in READINGS[1:]]
for r in READINGS:
    RN[(r, "legend")] = RN[r]
RN[(MAIN, "code")] = "main, code P1b"
P1b = {k: p1b_day(*k) for k in PB}
print("mice with a P1b:", {RN[k]: len(P1b[k]) for k in PB})
for k in PB[:2]:
    print(f"days from P1 to P1b ({RN[k]}):", (P1b[k] - P1[MAIN].p1_tday0.reindex(P1b[k].index))
          .describe()[["min", "50%", "max"]].to_dict())
print("main-reading mice with P1 but no 1b status (no code P1b possible):",
      int(P1[MAIN].b1_session.isna().sum()), "| with 1b but no biased session after it:",
      int(P1[MAIN].b1_session.notna().sum() - len(P1b[(MAIN, "code")])))

## fig3s5-first-full-sessions

> In the first three sessions of the full task, bias in each block, contrast threshold in each
> block and performance on 100% contrast trials did not vary significantly across laboratories,
> and a classifier could not predict lab membership.

In [ ]:
def first_full(r, how):
    if how == "legend: first three biased days":
        t = Tm[Tm.subject_id.isin(P1b[r].index) & (Tm.task == "biased")]
        t = t[t.tday0 >= t.subject_id.map(P1b[r])]
        firstdays = t.groupby("subject_id").tday0.apply(lambda x: sorted(x.unique())[:3])
        return t[[d in firstdays[s] for s, d in zip(t.subject_id, t.tday0)]]
    return ib.window(Tm, P1b[r], 1, 3, ["biased"])

ff = {}
rows = []
for r in PB:
    # the code's sessions, a variant, for the main reading of P1 only
    for how in ("legend: first three biased days", "code: days P1b+1 to P1b+3")[: 2 if r in PB[:2] else 1]:
        t = first_full(r, how)
        t = t.assign(block=np.select([np.isclose(t.probabilityLeft, 0.2), np.isclose(t.probabilityLeft, 0.8)],
                                     ["2080", "8020"], "5050"))
        fits = ib.mouse_fits(t, by=("subject_id", "block")).unstack("block")
        v = pd.DataFrame({f"{p}_{b}": fits[(p, b)] for p in ("bias", "threshold") for b in ("2080", "5050", "8020")})
        v["perf_100"] = t[t.abs_contrast == 100].groupby("subject_id").correct.mean() * 100
        v["perf_easy_code"] = t[t.abs_contrast >= 50].groupby("subject_id").correct.mean() * 100
        v["lab"] = lab[v.index]
        ff[(r, how)] = v
        ps = {c: ib.lab_test(v[c], v.lab, "kruskal") for c in v.columns if c not in ("lab", "perf_easy_code")}
        from statsmodels.stats.multitest import multipletests
        holm = multipletests(list(ps.values()), method="holm-sidak")[1]
        rows.append(dict(reading=RN[r], sessions=how, mice=len(v), **{k: round(x, 3) for k, x in ps.items()},
                         p_perf_easy_code=round(ib.lab_test(v.perf_easy_code, v.lab, "kruskal"), 3),
                         min_p=min(ps.values()), min_p_holm=holm.min(), all_ns=min(ps.values()) >= 0.05))
print(pd.DataFrame(rows).to_string())

The classifier part of this claim runs in the section on the lab classifiers below, which
gives its conclusion.

## fig3s5a-prior-used-from-first-sessions

> Mice incorporated the block priors into their choices from the first sessions of the full task.

In [ ]:
rows = []
for (r, how), v in ff.items():
    d = (v.bias_8020 - v.bias_2080).dropna()
    w = stats.wilcoxon(v.bias_8020, v.bias_2080, nan_policy="omit")
    rows.append(dict(reading=RN[r], sessions=how, n=len(d), median_diff=d.median(), positive=f"{int((d > 0).sum())}/{len(d)}",
                     p=w.pvalue, above_5=f"{int((d > 5).sum())}/{len(d)}", ok=d.median() > 0 and w.pvalue < 0.05))
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))

**Result: reproduced.** In the first full-task sessions bias is higher in 80:20 than in 20:80
blocks under every reading (main, legend sessions: median difference 4.45, 69 of 72 mice,
Wilcoxon p = 2.1e-13; over all readings the median difference is 4.35 to 6.40 and p is 2.0e-12 or
smaller). Between 30 of 73 and 39 of 66 mice exceed the proficiency threshold of 5, a variant.

## full-task-training-time

> Mice that reached full task proficiency took 31.5 +- 16.1 days, or 20,494 +- 10,980 trials
> (s.d., n = 98), from the start of training; the fastest took 8 days (4691 trials), the slowest
> 81 days (61,316 trials).

In [ ]:
nd = P2.p2_tday0 + 1
dd = Dc[Dc.subject_id.isin(nd.index)]
ntr = dd[dd.tday0 < dd.subject_id.map(nd)].groupby("subject_id").n_trials.sum()
EXP_FT = [(31.5, 3.15), (16.1, 1.61), (20494, 2049), (10980, 1098), (8, 0.8), (81, 8.1), (4691, 469), (61316, 6132)]
v = [nd.mean(), nd.std(), ntr.mean(), ntr.std(), nd.min(), nd.max(), ntr.min(), ntr.max()]
for (e, t), x, name in zip(EXP_FT, v, ["mean days", "sd days", "mean trials", "sd trials", "min days", "max days",
                                       "min trials", "max trials"]):
    print(f"{name}: {x:.1f} (paper {e}, within tolerance: {abs(x - e) <= t})")
print("mice:", len(nd), "| by lab:", lab[nd.index].value_counts().sort_index().to_dict())
active = Dc.groupby("subject_id").date.max() >= "2020-03-16"
print("last date of the dataset", Dc.date.max(), "| whole-training mice without P2 with a session in its last week:",
      int((active & ~active.index.isin(P2.index)).sum()))

**Result: partially reproduced.** The 60 mice that reach full task proficiency take 29.4 +- 14.1
days and 19,637 +- 9,173 trials: the means are within tolerance, the s.d.s are not (16.1 and
10,980 in the paper). The fastest takes 10 days (5,592 trials) and the slowest 64 days (43,500
trials), against 8 (4,691) and 81 (61,316). 30 mice without P2 were still training in the
dataset's last week, which cuts off slow learners.

## fig4cd-biasshift-largest-at-zero

> Block structure had the greatest effect on choices when sensory evidence was absent; the shift in
> rightward choices between 20:80 and 80:20 blocks was highest at 0% contrast in every laboratory,
> averaging 28.5%.

In [ ]:
CON = sorted(S2t.signed_contrast.unique())
def curve(row, b, c):
    p = {k: row[(k, b)] for k in ("bias", "threshold", "lapselow", "lapsehigh")}
    return 100 * ib.gamma_curve(p, c)
sh = pd.DataFrame({c: [curve(S2fits.loc[m], "2080", c) - curve(S2fits.loc[m], "8020", c) for m in S2fits.index]
                   for c in CON}, index=S2fits.index)
print("signed contrasts:", CON)
print("mean shift over mice by contrast:", sh.mean().round(2).to_dict())
by_lab = sh.groupby(lab[sh.index]).mean()
print(by_lab.round(1).to_string())
at0 = sh[0.0]
print(f"mean shift at 0%: {at0.mean():.2f} (n = {at0.notna().sum()}), within 28.5 +- 2.85: {abs(at0.mean() - 28.5) <= 2.85}")
print("labs whose mean shift is highest at 0%:", int((by_lab.idxmax(axis=1) == 0.0).sum()), "of", len(by_lab),
      "| argmax per lab:", by_lab.idxmax(axis=1).to_dict())

**Result: reproduced.** The mean shift in fitted rightward choices between 20:80 and 80:20 blocks
is 28.00% at 0% contrast (60 mice; within 28.5 +- 2.85), and in each of the 7 labs the lab mean is
highest at 0%.

## fig4de-biasshift-no-lab-difference

> The bias shift at 0% contrast did not differ across laboratories more than within them.

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
dfa = pd.DataFrame({"shift": at0, "lab": lab[at0.index]}).dropna()
aov = sm.stats.anova_lm(smf.ols("shift ~ C(lab)", data=dfa).fit(), typ=2)
print(aov)
print("p >= 0.05:", aov["PR(>F)"].iloc[0] >= 0.05, "| lab means:", dfa.groupby("lab")["shift"].mean().round(1).to_dict())

**Result: reproduced.** The one-way ANOVA of the shift at 0% across labs gives F = 2.07, p =
0.072 (paper F = 1.345, p = 0.2455), with lab means from 23.0 to 36.2%.

## fig4f, fig4gh, fig4i: block effects on the psychometric parameters

> Block type did not significantly affect the contrast threshold.

> Block type changed lapse rates, which were higher on the left in 20:80 blocks and higher on the
> right in 80:20 blocks.

> Block type changed the overall bias consistently, with curves shifting left in 20:80 blocks and
> right in 80:20 blocks.

In [ ]:
def wil(a, b):
    d = (a - b).dropna()
    return d.median(), stats.wilcoxon(a, b, nan_policy="omit").pvalue, int((d > 0).sum()), len(d)
md_, p, npos, n = wil(S2fits[("threshold", "2080")], S2fits[("threshold", "8020")])
print(f"threshold 20:80 - 80:20: median {md_:.2f}, {npos}/{n} positive, Wilcoxon p = {p:.4g}; p >= 0.05: {p >= 0.05}")

**Result: reproduced.** The threshold does not differ between 20:80 and 80:20 blocks (median
difference 0.08, 31 of 60 positive, Wilcoxon p = 0.80; paper p = 0.85).

In [ ]:
m1, p1, n1, n = wil(S2fits[("lapselow", "2080")], S2fits[("lapselow", "8020")])
m2, p2, n2, _ = wil(S2fits[("lapsehigh", "8020")], S2fits[("lapsehigh", "2080")])
print(f"left lapse, 20:80 - 80:20: median {m1:.4f}, {n1}/{n} positive, p = {p1:.3g}")
print(f"right lapse, 80:20 - 20:80: median {m2:.4f}, {n2}/{n} positive, p = {p2:.3g}")
print("directions hold:", m1 > 0 and m2 > 0, "| p < 1e-6 and < 1e-7:", p1 < 1e-6 and p2 < 1e-7,
      "| both p < 0.05:", p1 < 0.05 and p2 < 0.05)
print("smallest p a Wilcoxon test can give with n = %d (normal approximation, all differences one sign): %.2g"
      % (n, stats.wilcoxon(np.arange(1, n + 1)).pvalue))

**Result: partially reproduced.** The left lapse is higher in 20:80 blocks (median difference
0.015, 46 of 60 mice, p = 2.3e-06) and the right lapse higher in 80:20 blocks (0.018, 45 of 60, p =
2.2e-05). The directions hold with p < 0.05, but not with p < 1e-6 and < 1e-7.

In [ ]:
m3, p3, n3, n = wil(S2fits[("bias", "8020")], S2fits[("bias", "2080")])
print(f"bias 80:20 - 20:80: median {m3:.2f}, {n3}/{n} positive, p = {p3:.3g}")
print("direction holds:", m3 > 0, "| p < 1e-16:", p3 < 1e-16, "| p < 0.05:", p3 < 0.05)

**Result: partially reproduced.** Bias is higher in 80:20 than in 20:80 blocks in all 60 mice
(median difference 7.11, p = 1.6e-11). This is the smallest p the test gives with 60 mice, so the
paper's p < 1e-16 (98 mice) cannot be reached, and the criterion gives partial.

## fig4fi-no-lab-difference

> Threshold, lapses and bias in each block type showed no significant differences across
> laboratories.

In [ ]:
v8 = pd.DataFrame({f"{p}_{b}": S2fits[(p, b)] for p in ("threshold", "lapselow", "lapsehigh", "bias")
                   for b in ("8020", "2080")})
pv = np.array([ib.lab_test(v8[c], lab[v8.index], "code") for c in v8.columns])
pk = np.array([ib.lab_test(v8[c], lab[v8.index], "kruskal") for c in v8.columns])
print(pd.DataFrame({"code_switch": pv, "code_switch_fdr": ib.fdr(pv), "kruskal_fdr": ib.fdr(pk)}, index=v8.columns).round(4))
print("all eight corrected p >= 0.05:", bool((ib.fdr(pv) >= 0.05).all()))

**Result: reproduced.** None of the eight per-block parameters differs across labs after
correction (corrected p 0.19 to 0.88).

## Lab classifiers (fig3f-classifier-chance-basic, fig3f-positive-control, fig3s3-other-classifiers, fig4j-classifier-chance-full, and the classifier parts of fig3s4-looser-criterion and fig3s5-first-full-sessions)

> A Naive Bayes classifier could not identify a mouse's laboratory from its basic-task performance,
> threshold and bias; its accuracy was 0.12 +- 0.051 against a chance level of 0.11, and the 95%
> range of its distribution included chance.

> With the laboratory's time zone added as a feature, the classifier performed above chance and
> confused labs in the same time zone.

> Random Forest and Logistic Regression classifiers also failed to predict laboratory membership
> from basic-task behavior.

> A classifier trained on the full-task psychometric parameters could not predict the laboratory
> of individual mice above chance.

The classifier draws the same number of mice from each lab, 2000 times, with leave-one-out
cross-validation and a null from shuffled lab labels (claims file header, Lab classifier). The
subsample size follows the Methods' rule (answer of 2026-10-22): the lowest number of mice per lab
over all the datasets classified, one size for every classifier claim. The datasets are those of
the main reading: S1 (fig3f, fig3s3), S2 (fig4j), the looser criterion (fig3s4) and the first
full-task sessions (fig3s5). The code's 8 mice per lab is a variant. A classifier is above chance
when the 2.5th percentile of its accuracies is above the mean shuffled accuracy.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
F_S1 = ["perf_easy", "threshold", "bias"]
F_FF = ["bias_2080", "bias_5050", "bias_8020", "threshold_2080", "threshold_5050", "threshold_8020", "perf_100"]
LEGEND = "legend: first three biased days"
CLF_DATA = {
    "S1": S1[MAIN][S1[MAIN].rt.notna()].dropna(subset=F_S1),
    "S2": v8.dropna().assign(lab=lab[v8.dropna().index].to_numpy()),
    "looser criterion": vl.dropna(subset=F_S1),
    "first full-task sessions": ff[((MAIN, "legend"), LEGEND)].dropna(subset=F_FF),
}
clf_counts = pd.DataFrame({k: v.lab.value_counts() for k, v in CLF_DATA.items()}).reindex(ib.LABS).fillna(0).astype(int)
print(clf_counts.to_string())
K = int(clf_counts.min().min())
print("mice per lab drawn (lowest count over the datasets):", K,
      "| datasets with a lab under 8 (the code's size cannot run):", list(clf_counts.columns[clf_counts.min() < 8]))
# The direct leave-one-out classifiers against scikit-learn, on subsamples of S1 with true and shuffled labels
rng_chk = np.random.default_rng(5)
d = CLF_DATA["S1"]
X1, y1 = d[F_S1].to_numpy(), d.lab.to_numpy()
agree_nb, agree_lr = [], []
for k in range(20):
    idx = np.concatenate([rng_chk.choice(np.flatnonzero(y1 == l), K, replace=False) for l in ib.LABS])
    ys = y1[idx] if k % 2 else rng_chk.permutation(y1[idx])
    agree_nb.append(np.mean(ib.loo_gaussian_nb(X1[idx], ys) == cross_val_predict(GaussianNB(), X1[idx], ys, cv=LeaveOneOut())))
    agree_lr.append(np.mean(ib.loo_ovr_logistic(X1[idx], ys) == cross_val_predict(
        OneVsRestClassifier(LogisticRegression(solver="liblinear")), X1[idx], ys, cv=LeaveOneOut())))
print(f"agreement with scikit-learn over 20 subsamples: GaussianNB {np.mean(agree_nb):.4f}, "
      f"one-vs-rest liblinear LogisticRegression {np.mean(agree_lr):.4f}")

def run_clf(name, X, labs, kind="nb", n_iter=2000, **kw):
    out = ib.classify_labs(X, labs, kind, n_iter=n_iter, per_lab=K, **kw)
    acc, null = out[0], out[1]
    row = dict(classifier=name, mice=len(labs), per_lab=K, n_iter=n_iter, acc_mean=acc.mean(), acc_sd=acc.std(),
               acc_p2_5=np.percentile(acc, 2.5), acc_p97_5=np.percentile(acc, 97.5), null_mean=null.mean(),
               above_chance=np.percentile(acc, 2.5) > null.mean())
    return row, out

CLF = {}
rows = []
for name, data, feats, kind in (("fig3f: Naive Bayes, S1", "S1", F_S1, "nb"),
                                ("fig3s3: Logistic Regression, S1", "S1", F_S1, "lr"),
                                ("fig4j: Naive Bayes, S2", "S2", list(v8.columns), "nb"),
                                ("fig3s4: Naive Bayes, looser criterion", "looser criterion", F_S1, "nb"),
                                ("fig3s5: Naive Bayes, first full-task sessions", "first full-task sessions", F_FF, "nb")):
    d = CLF_DATA[data]
    row, CLF[name] = run_clf(name, d[feats].to_numpy(), d.lab.to_numpy(), kind)
    rows.append(row)
d = CLF_DATA["S1"]
Xtz = np.column_stack([d[F_S1].to_numpy(), d.lab.map(ib.TIME_ZONE).to_numpy()])
row, CLF["fig3f-positive-control"] = run_clf("fig3f-positive-control: Naive Bayes, S1 + time zone", Xtz, d.lab.to_numpy(),
                                             "nb", return_confusion=True)
rows.append(row)
clf_tab = pd.DataFrame(rows).set_index("classifier")
print(clf_tab.round(3).to_string())

In [ ]:
# Positive control: the code's second threshold, and the mean confusion matrix (rows: true lab)
acc_pc, null_pc, names, conf = CLF["fig3f-positive-control"]
print(f"positive control, 0.001th percentile {np.percentile(acc_pc, 0.001):.3f} against the null mean {null_pc.mean():.3f}")
print(pd.DataFrame(conf, index=names, columns=names).round(3).to_string())
tz = np.array([ib.TIME_ZONE[l] for l in names])
same = (tz[:, None] == tz[None, :]) & ~np.eye(len(names), dtype=bool)
other = tz[:, None] != tz[None, :]
print(f"off-diagonal mass, same time zone: total {conf[same].sum():.3f} over {same.sum()} cells (mean {conf[same].mean():.4f}); "
      f"other time zones: total {conf[other].sum():.3f} over {other.sum()} cells (mean {conf[other].mean():.4f})")
print("diagonal (correct) by lab:", dict(zip(names, np.round(np.diag(conf), 3))))

In [ ]:
# Random Forest (fig3s3): 100 trees per fit, 35 fits per subsample and 2000 subsamples with as many
# shuffles. Timed on a few subsamples; the result over these decides nothing.
N_RF = 6
t_rf = time.time()
row_rf, CLF["fig3s3: Random Forest, S1"] = run_clf("fig3s3: Random Forest, S1", X1, y1, "rf", n_iter=N_RF)
per_sub = (time.time() - t_rf) / N_RF
print(f"Random Forest: {per_sub:.1f} s per subsample (with its shuffle); 2000 subsamples would take {2000 * per_sub / 3600:.1f} h")
print(pd.Series(row_rf).to_string())

In [ ]:
# Variant: the code's 8 mice per lab, where every lab of a dataset holds 8 or more
for data in CLF_DATA:
    short = list(clf_counts.index[clf_counts[data] < 8])
    print(f"{data}: 8 mice per lab {'cannot run, fewer in ' + ', '.join(short) if short else 'can run'}")
# Variant: Naive Bayes on S1 under the other P1 readings, with the same subsample size
for r in READINGS[1:]:
    d = S1[r][S1[r].rt.notna()].dropna(subset=F_S1)
    if d.lab.value_counts().reindex(ib.LABS).fillna(0).min() < K:
        print(RN[r], "has a lab with fewer than", K, "mice")
        continue
    row, _ = run_clf(f"Naive Bayes, S1, {RN[r]}", d[F_S1].to_numpy(), d.lab.to_numpy(), "nb")
    print(f"{RN[r]}: accuracy {row['acc_mean']:.3f} +- {row['acc_sd']:.3f}, 2.5th percentile {row['acc_p2_5']:.3f}, "
          f"null mean {row['null_mean']:.3f}, above chance {row['above_chance']}")

**Result: reproduced.** The subsample is 5 mice per lab, the lowest lab count over the four
datasets classified (Lab 4 and Lab 7 in S2, Lab 7 in the first full-task sessions). The Naive
Bayes classifier on S1 (81 mice) has accuracy 0.140 +- 0.063, within 0.12 +- 0.051, against a
shuffled mean of 0.106, within 0.11 +- 0.011. The 2.5th percentile of its accuracies (0.029) is
below the shuffled mean, so it is not above chance, as in the paper. Under the pipeline's contrast
check the accuracy is 0.138 and the 2.5th percentile 0.029 (null 0.110); the 1b-replace readings
leave a lab with fewer than 5 mice. The code's 8 mice per lab cannot run, as Lab 4 and Lab 7 hold 7
and 6 mice in S1.

**Result: reproduced.** With the lab's time zone as a fourth feature the accuracy is 0.405 +-
0.084, and both the 2.5th percentile (0.257) and the code's 0.001th percentile (0.171) are above
the shuffled mean of 0.118. In the mean confusion matrix all off-diagonal mass lies on labs in the
same time zone (4.166 over 12 cells, against 0.000 over the 30 cells of other zones); Lab 7, alone
in its zone, is always classified correctly.

**Result: blocked (not_runnable).** The Logistic Regression classifier (liblinear, one-vs-rest,
which agrees with scikit-learn on every subsample checked) is not above chance: accuracy 0.071 +-
0.054, 2.5th percentile 0.000 against a shuffled mean of 0.070. The Random Forest takes 7.9 s per
subsample here, about 4.4 h for the 2000 subsamples the criterion asks, beyond the time limit of
the notebook. Over 6 subsamples, which decide nothing, its accuracy is 0.105 against a shuffled
mean of 0.119.

**Result: reproduced.** On the eight per-block values of S2 (60 mice), with 5 mice per lab, the
Naive Bayes classifier has accuracy 0.176 +- 0.046, and its 2.5th percentile (0.086) is below the
shuffled mean of 0.107, so it is not above chance. The code's 8 mice per lab cannot run, as five
labs hold fewer at full task proficiency.

**Result: not reproduced.** 91 mice meet the looser criterion, against 150 +- 15. Trials per
session differ across labs (corrected p = 0.0022), as in the paper, but so does the number of
trials per minute (corrected p = 0.0022), where the paper found no difference. The Naive Bayes
classifier, 5 mice per lab, is not above chance, as in the paper (accuracy 0.117 +- 0.058, 2.5th
percentile 0.029 against a shuffled mean of 0.110).

**Result: not reproduced.** Under the main reading (legend sessions from P1b, the first full-task
session at or after P1; 72 mice) the Kruskal-Wallis p is below 0.05 for the threshold in 20:80
blocks (0.022) and in 50:50 blocks (0.038) and for performance on 100% contrast (0.022); the
paper's seven p values are 0.078 to 0.96. Every variant also has a p below 0.05 (smallest 0.001 to
0.025, with the code's P1b 0.007); with the code's Holm-Sidak correction, a variant, the smallest
is 0.141 under the main reading. The Naive Bayes classifier on the seven variables, 5 mice per lab,
is not above chance (accuracy 0.170 +- 0.048, 2.5th percentile 0.086 against a shuffled mean of
0.114).

## Probabilistic choice model (fig5cf, fig5dg, fig5s2)

Sessions per the code: training days P1b-2 to P1b+3, split by protocol into basic and full; or
(b) the last three basic days before P1b and the first three full days. The 50% contrast, which the
paper does not settle for a mouse shown no 50% trials (answer of 2026-10-22): reading (a), the
Methods' model with the 50% weight left out for that mouse (`drop50=False`); reading (b), the
code's model, 50% trials dropped for every mouse (`drop50=True`).

In [ ]:
def glm_trials(r, how):
    if how == "code: P1b-2..P1b+3":
        t = ib.window(Tm, P1b[r], -2, 3, ["training", "biased"])
        return t[t.task == "training"], t[t.task == "biased"]
    tb = ib.window(Tm, P1b[r], -60, -1, ["training"])
    lastdays = tb.groupby("subject_id").tday0.apply(lambda x: sorted(x.unique())[-3:])
    tb = tb[[d in lastdays[s] for s, d in zip(tb.subject_id, tb.tday0)]]
    tf = ib.window(Tm, P1b[r], 0, 60, ["biased"])
    firstdays = tf.groupby("subject_id").tday0.apply(lambda x: sorted(x.unique())[:3])
    tf = tf[[d in firstdays[s] for s, d in zip(tf.subject_id, tf.tday0)]]
    return tb, tf

GLM = {}
for r in PB:
    for how in ("code: P1b-2..P1b+3", "(b) last 3 basic, first 3 full"):
        tb, tf = glm_trials(r, how)
        for drop50 in (False, True):
            items, keys = [], []
            for task, t in (("basic", tb), ("full", tf)):
                for sid, d in t.groupby("subject_id"):
                    items.append((ib.glm_design(d, task == "full", drop50), task == "full", drop50))
                    keys.append((sid, task))
            res = ib.fit_glms(items)
            df = pd.DataFrame(res, index=pd.MultiIndex.from_tuples(keys, names=["subject_id", "task"]))
            GLM[(r, how, drop50)] = df
print("fits:", {f"{RN[k[0]]} | {k[1]} | drop50={k[2]}": f"{len(v)} ({int(v.get('error', pd.Series(dtype=object)).notna().sum())} failed)"
                for k, v in GLM.items()})

## fig5cf-glm-weights

> In a logistic choice model fitted to each mouse, visual weights grew with contrast to above 3 at
> 100% contrast, while weights for past choices were much smaller and positive after both rewarded
> (basic 0.19, full 0.42) and unrewarded (basic 0.33, full 0.46) trials, indicating perseverance.

In [ ]:
EXP_W = [(0.19, 0.019), (0.42, 0.042), (0.33, 0.033), (0.46, 0.046)]
rows = []
for (r, how, drop50), df in GLM.items():
    m = df.groupby(level="task").mean(numeric_only=True)
    cols = [c for c in ["c6_25", "c12_5", "c25_0", "c50_0", "c100_0"] if c in m.columns]
    inc = all(np.all(np.diff(m.loc[t, cols].dropna().to_numpy()) > 0) for t in ("basic", "full"))
    w = [m.loc["basic", "rewarded"], m.loc["full", "rewarded"], m.loc["basic", "unrewarded"], m.loc["full", "unrewarded"]]
    rows.append(dict(reading=RN[r], sessions=how, drop50=drop50,
                     n=f"{(df.index.get_level_values('task') == 'basic').sum()},{(df.index.get_level_values('task') == 'full').sum()}",
                     with_50=int(df.get("c50_0", pd.Series(dtype=float)).notna().sum()),
                     w100_basic=m.loc["basic", "c100_0"], w100_full=m.loc["full", "c100_0"], increasing=inc,
                     rew_basic=w[0], rew_full=w[1], unrew_basic=w[2], unrew_full=w[3],
                     n_hist_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(w, EXP_W)),
                     ok=m.loc["basic", "c100_0"] > 3 and m.loc["full", "c100_0"] > 3 and inc and check(w, EXP_W)))
glm_tab = pd.DataFrame(rows)
print(glm_tab.round(4).to_string())

**Result: partially reproduced.** Under the main reading of P1b the 100% contrast weight is above 3
in both tasks (3.48 to 3.83) and the past-choice weights are positive, but no reading has all
three parts. On the code's sessions with reading (a), the 50% weight kept for the 7 of 70 mice
shown 50% trials in the basic task, all four history weights are within tolerance (rewarded 0.202
and 0.430, unrewarded 0.297 and 0.440, against 0.19, 0.42, 0.33 and 0.46) but the mean visual
weights do not increase with contrast; with reading (b), 50% trials dropped, they increase and the
basic-task unrewarded weight falls just below 0.297, the edge of its tolerance. With the code's
P1b, a variant, the sessions of reading (b) meet all three parts under both 50% readings (rewarded
0.201 to 0.202 and 0.431, unrewarded 0.345 to 0.347 and 0.415), as does the pipeline's contrast
check on the code's sessions with 50% trials dropped.

## fig5dg-history-weights-higher-full

> Previous choices influenced behavior more strongly in the full task than in the basic task,
> after both rewarded and unrewarded trials.

In [ ]:
rows = []
for (r, how, drop50), df in GLM.items():
    w = df[["rewarded", "unrewarded"]].unstack("task").dropna()
    tr = stats.ttest_rel(w[("rewarded", "full")], w[("rewarded", "basic")])
    tu = stats.ttest_rel(w[("unrewarded", "full")], w[("unrewarded", "basic")])
    rows.append(dict(reading=RN[r], sessions=how, drop50=drop50, n=len(w), t_rew=tr.statistic, p_rew=tr.pvalue,
                     t_unrew=tu.statistic, p_unrew=tu.pvalue,
                     ok=tr.statistic > 0 and tu.statistic > 0 and tr.pvalue < 0.05 and tu.pvalue < 0.05))
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))

**Result: reproduced.** On the code's sessions both history weights are larger in the full task
under every reading and both 50% readings (main: rewarded t = 9.91 and 9.94, unrewarded t = 3.54
and 3.56, p < 0.001; over all readings rewarded t = 8.91 to 10.69 and unrewarded t = 3.54 to
3.79). On the sessions of reading (b) the unrewarded difference is not significant (p = 0.10 to
0.39).

## fig5s2-glm-accuracy

> The model predicted individual mouse choices well, with cross-validated accuracy of
> 81.03 +- 5.1% in the basic task and 82.1 +- 5.8% in the full task, similar across
> laboratories; mean condition numbers were 2.4 and 3.2.

In [ ]:
EXP_A = [(81.03, 8.1), (82.1, 8.2), (2.4, 0.24), (3.2, 0.32)]
rows = []
for (r, how, drop50), df in GLM.items():
    m = df.groupby(level="task").mean(numeric_only=True)
    s = df.groupby(level="task").std(numeric_only=True)
    for acc in ("acc_threshold", "acc_prob"):
        v = [m.loc["basic", acc], m.loc["full", acc], m.loc["basic", "cond"], m.loc["full", "cond"]]
        rows.append(dict(reading=RN[r], sessions=how, drop50=drop50, accuracy=acc,
                         basic=f"{v[0]:.1f} +- {s.loc['basic', acc]:.1f}", full=f"{v[1]:.1f} +- {s.loc['full', acc]:.1f}",
                         cond_basic=v[2], cond_full=v[3],
                         n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_A)), ok=check(v, EXP_A)))
print(pd.DataFrame(rows).round(2).to_string())
# spread over fold seeds, one reading
tb, tf = glm_trials(PB[0], "code: P1b-2..P1b+3")
seed_acc = []
for seed in range(5):
    items = [(ib.glm_design(d, False, True), False, True, 5, seed) for _, d in tb.groupby("subject_id")]
    seed_acc.append(np.mean([x["acc_threshold"] for x in ib.fit_glms(items) if "error" not in x]))
print("basic-task accuracy (main, code sessions, no 50%) over fold seeds 0-4:", np.round(seed_acc, 2))

**Result: blocked (methods_underspecified).** Under the main reading, on the code's sessions, the
cross-validated accuracy (legend reading) is 81.9 +- 3.8% in the basic task and 83.5 +- 3.0% in
the full task, within tolerance, and the full-task condition number is 3.14, within 3.2 +- 0.32.
The basic-task condition number turns on the 50% weight, which the paper does not settle: 2.38
with 50% trials dropped (reading (b), within 2.4 +- 0.24) and 2.89 with the 50% weight kept for the
mice shown 50% trials (reading (a), outside). The two readings give different outcomes. The code's
accuracy reading gives 74.3% and 76.4%, also within tolerance, and accuracy varies by less than
0.1% over fold seeds.

## fig5s2-weights-similar-across-labs

> Fitted model weights and predictive accuracy were similar across laboratories.

In [ ]:
rows = []
for (r, how, drop50), df in GLM.items():
    ps = {}
    for task in ("basic", "full"):
        d = df.xs(task, level="task")
        cols = ["c6_25", "c12_5", "c25_0", "c100_0", "rewarded", "unrewarded", "Intercept"] + (["block"] if task == "full" else []) \
            + ["pseudo_r2", "acc_threshold"]
        for c in cols:
            ps[f"{task}:{c}"] = ib.lab_test(d[c], lab[d.index], "code")
    ps = pd.Series(ps)
    rows.append(dict(reading=RN[r], sessions=how, drop50=drop50, tests=len(ps), n_sig=int((ps < 0.05).sum()),
                     sig=", ".join(f"{k} {v:.3f}" for k, v in ps[ps < 0.05].items()), min_p=ps.min()))
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))

**Result: not reproduced.** Under the main reading 6 (code's sessions) and 7 (sessions of reading
(b)) of the 19 lab tests give p < 0.05, under both 50% readings, mostly the visual weights (6.25%
contrast in both tasks, p = 0.001 to 0.046), where the criterion allows at most two for partial.
With the code's P1b, a variant, 2 and 4 tests do; under the other variants 2 to 8.

## fig5s1b-win-stay-lose-switch

> After correcting for slow fluctuations in decision bound, most mice used a win-stay lose-switch
> strategy, in both the basic and full tasks.

In [ ]:
def history_shift(t, cond_prefix, extra=None):
    """Fitted rightward % at 0% after a right minus after a left choice (of the previous or next
    trial), separately after rewarded and unrewarded trials; fits with more than 50 trials."""
    out = {}
    pairs, keys = [], []
    for (sid, rew, side), d in t.groupby(["subject_id", f"{cond_prefix}_reward", f"{cond_prefix}_choice"]):
        if side == 0 or len(d) <= 50:
            continue
        pairs.append((d.signed_contrast.to_numpy(), d.choice_right.to_numpy()))
        keys.append((sid, rew, side))
    fits = ib.fit_many(pairs)
    f = pd.DataFrame(fits, index=pd.MultiIndex.from_tuples(keys, names=["subject_id", "rew", "side"]))
    p0 = f.apply(lambda row: 100 * ib.gamma_curve(row, 0.0) if row.notna().all() else np.nan, axis=1)
    p0 = p0.unstack("side")
    return (p0.get(1.0) - p0.get(-1.0)).unstack("rew")

HIST = {}
def strategy(r, by_contrast=False):
    if r not in HIST:
        tt = ib.history_columns(ib.window(Tm, P1b[r], -2, 3, ["training", "biased"]))
        HIST[r] = tt[tt.choice_right.notna()]
    tt = HIST[r]
    res = {}
    for task, t in (("basic", tt[tt.task == "training"]), ("full", tt[tt.task == "biased"])):
        n_by_c = t.signed_contrast.value_counts()
        t = t[t.signed_contrast.isin(n_by_c[n_by_c >= 100].index)]
        if not by_contrast:
            res[task] = history_shift(t, "prev") - history_shift(t, "next")
        else:
            out = {}
            for name, cs in (("0", [0.0]), ("6.25", [6.25]), ("0+6.25", [0.0, 6.25]), ("100", [100.0])):
                tp = t[t.prev_contrast.isin(cs) & (t.prev_reward == 1)]
                tn = t[t.next_contrast.isin(cs) & (t.next_reward == 1)]
                out[name] = (history_shift(tp, "prev") - history_shift(tn, "next")).get(1.0)
            res[task] = pd.DataFrame(out)
    return res

rows = []
STRAT = {}
for r in PB[:2]:
    STRAT[r] = strategy(r)
    for task, c in STRAT[r].items():
        c = c.dropna()
        win, lose = c[1.0], c[0.0]
        rows.append(dict(reading=RN[r], task=task, mice=len(c), post_reward_median=win.median(), post_error_median=lose.median(),
                         frac_win_stay=(win > 0).mean(), frac_lose_switch=(lose < 0).mean(),
                         frac_both=((win > 0) & (lose < 0)).mean()))
ws = pd.DataFrame(rows)
print(ws.round(3).to_string())
print("\nmore than half with both, in each task, per reading:",
      ws.groupby("reading").frac_both.apply(lambda x: bool((x > 0.5).all())).to_dict())

**Result: reproduced.** After correction, more than half the mice both stay after a reward and
switch after an error, in each task (main: basic 0.587 of 63 mice, full 0.859 of 71; code P1b:
0.596 and 0.833). The median corrected shift in the basic task is 4.77% after rewards and -8.94%
after errors (main). The P1 variants are not run for this claim, to keep the notebook within its
time limit.

## fig5s1c-updating-by-contrast

> After rewarded trials, choice updating was largest when the previous stimulus had low contrast
> and much smaller after high-contrast rewarded trials.

In [ ]:
rows = []
for r in PB[:2]:
    res = strategy(r, by_contrast=True)
    for task, d in res.items():
        m = d.mean()
        rows.append(dict(reading=RN[r], task=task, **{f"n_{k}": int(d[k].notna().sum()) for k in d.columns},
                         **{f"mean_{k}": m[k] for k in d.columns},
                         each_above_100=m["0"] > m["100"] and m["6.25"] > m["100"], pooled_above_100=m["0+6.25"] > m["100"]))
uc = pd.DataFrame(rows)
print(uc.round(2).to_string())
print("\nper reading, both tasks:", uc.groupby("reading")[["each_above_100", "pooled_above_100"]].all().to_dict())

**Result: reproduced.** After rewarded trials the mean corrected updating after 0% and after 6.25%
contrast exceeds that after 100% in both tasks, under the main reading (full task: 11.35 and 8.44
against 6.73%; basic task: 15.48 and 10.87 against 5.13%) and with the code's P1b. In the basic
task the 0% value rests on 4 mice, as few 0% trials pass the 50-trial rule; with 0% and 6.25%
pooled the result is the same (11.13 against 5.13%).

## session-end-criteria

> The session-ending criterion was not mandatory; 5,316 of 10,903 sessions ended earlier and
> 5,587 later, the latter continuing for a median of 19 +- 16 extra trials (m.a.d.).

End status from the rig settings (`SUBJECT_DISENGAGED_TRIGGERED`, `_TRIALNUM`); for sessions
without one, recomputed from the three criteria of the Figure 1, figure supplement 2 legend.

In [ ]:
def end_table(sel):
    s = S[S.subject_id.isin(sel)].copy()
    recompute = s.disengaged.isna()
    trig = s.disengaged_trial.where(s.disengaged.isin([1, 2, 3]))
    tt = T[T.session_id.isin(s.session_id[recompute])]
    rc = {}
    for sid, d in tt.groupby("session_id"):
        d = d.sort_values("start_time")
        rc[sid] = ib.end_criterion_trial(d.stimOn_times.to_numpy(), d.rt.to_numpy(), d.start_time.iloc[0])
    s["trigger"] = trig
    s.loc[recompute, "trigger"] = s.loc[recompute, "session_id"].map(rc)
    s["met"] = s.trigger.notna()
    s["extra"] = s.n_trials - s.trigger
    return s, int(recompute.sum())

for name, sel in (("all project mice", mice), ("whole-training mice", complete)):
    s, nre = end_table(sel)
    day = s.groupby(["subject_id", "date"]).agg(met=("met", "any"), dur=("duration_min", "sum"))
    day_met = day.met | (day.dur >= 90)
    ex = s.extra[s.met]
    print(f"{name}: sessions {len(s)} (recomputed {nre}), met an end criterion {int(s.met.sum())}, ended earlier {int((~s.met).sum())}")
    print(f"   training days {len(day)}, met (status or 90 min) {int(day_met.sum())}")
    print(f"   extra trials: median {ex.median():.1f}, m.a.d. {stats.median_abs_deviation(ex.dropna()):.1f}, "
          f"negative {int((ex < 0).sum())}; without negatives median {ex[ex >= 0].median():.1f}, "
          f"m.a.d. {stats.median_abs_deviation(ex[ex >= 0].dropna()):.1f}")
    print(f"   criterion of the trigger: {s.disengaged[s.disengaged.isin([1, 2, 3])].value_counts().sort_index().to_dict()}")
print(f"\nruntime so far {(time.time() - T0) / 60:.1f} min")

**Result: not reproduced.** The project mice have 6,539 sessions, against 10,903. 5,698 sessions
met an end criterion and 841 ended earlier (13%), where the paper has 5,587 and 5,316 (49%). The
median number of extra trials is 16.0 (m.a.d. 14.0), against 19 +- 1.9 and 16 +- 1.6. Only the
count of sessions meeting a criterion is within tolerance, and the shares differ.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| cohort-counts | not reproduced |
| fig2g-days-to-proficiency | partially reproduced |
| fig2g-days-differ-across-labs | not reproduced |
| fig2s1-trials-to-proficiency | partially reproduced |
| fig2h-proportion-learned-40-days | reproduced |
| fig2bc-threshold-bias-iqr | partially reproduced |
| fig2-abs-bias-threshold-during-training | reproduced |
| fig2s2-variability-decreases | partially reproduced |
| early-sessions-predict-slow-learners | blocked: methods underspecified |
| fig3c-easy-errors-basic | partially reproduced |
| fig3de-threshold-bias-basic | reproduced |
| fig3cde-no-lab-difference-medians | reproduced |
| fig3cde-no-lab-difference-variances | reproduced |
| fig3s2a-trial-duration | not reproduced |
| fig3s2b-trials-per-session | partially reproduced |
| fig3f-classifier-chance-basic | reproduced |
| fig3f-positive-control | reproduced |
| fig3s3-other-classifiers | blocked: not runnable (Random Forest) |
| lab-link | reproduced |
| fig3s1a-choice-consistency-basic | reproduced |
| fig3s1bc-choice-consistency-full | not reproduced |
| fig3s4-looser-criterion | not reproduced |
| fig3s5-first-full-sessions | not reproduced |
| fig3s5a-prior-used-from-first-sessions | reproduced |
| full-task-training-time | partially reproduced |
| fig4cd-biasshift-largest-at-zero | reproduced |
| fig4de-biasshift-no-lab-difference | reproduced |
| fig4f-threshold-unchanged-by-block | reproduced |
| fig4gh-lapses-change-with-block | partially reproduced |
| fig4i-bias-shifts-with-block | partially reproduced |
| fig4fi-no-lab-difference | reproduced |
| fig4j-classifier-chance-full | reproduced |
| fig5cf-glm-weights | partially reproduced |
| fig5dg-history-weights-higher-full | reproduced |
| fig5s2-glm-accuracy | blocked: methods underspecified |
| fig5s2-weights-similar-across-labs | not reproduced |
| fig5s1b-win-stay-lose-switch | reproduced |
| fig5s1c-updating-by-contrast | reproduced |
| session-end-criteria | not reproduced |

How the analysis differs from the paper's:

- The dataset holds sessions from 2019-06-11 to 2020-03-23 only; the paper's mice began training
  in January 2019. Only the 136 project mice whose first session in the dataset is a first-stage
  session are used, against 206 that started training; every count and every test has fewer mice
  than the paper's.
- The files carry no training status. Basic task proficiency is recomputed from the trials as the
  pipeline's SessionTrainingStatus computes it (claims file header, answers of 2026-10-22), with
  "all contrasts introduced" required of each of the three sessions; full task proficiency is
  recomputed with Appendix 1, table 1d. The first full-task session follows P1 by 1 to 3 days for
  31 of 73 mice under the main reading.
- P1b is the first full-task session at or after P1 (main reading); the code's P1b, at or after
  the first session meeting the 1b criteria, is a variant.
- The classifiers draw 5 mice per lab, the lowest lab count over the datasets classified, against
  8 in the code. The Random Forest of fig3s3 was run on 6 subsamples only, as 2000 would take
  about 4.4 h. Logistic Regression is fitted one-vs-rest, as scikit-learn did with the liblinear
  solver when the paper's code ran.
- The GLM is run with the 50% weight left out for mice shown no 50% trials (reading (a)) and with
  50% trials dropped for every mouse (reading (b), the code's).

What was found about the dataset:

- It holds part of the paper's data: 178 mice and 6,615 sessions from 2019-06-11 on, against 206
  mice and 10,903 sessions from January 2019.
- One recording is deposited twice, under two mice (CSHL_013 and CSHL_015, 2019-07-15).
- The training status of each session, which the paper's analyses select on, is not in the
  files.
- `probabilityLeft` is described as the block's probability of a left stimulus, but in every
  basic-task session it varies from trial to trial between 0 and 1.
- `session_start_time` is stated in UTC (+00:00) but holds local clock times.
- 257,021 trials (6%) have no stimulus onset time, and 185 sessions have no onset time on any
  trial; 1,541 trials in 220 sessions have a response before the stimulus onset (down to
  -5,638 s). Trial durations, and the 1b reaction-time criterion, cannot be computed where onsets
  are missing.
- One session (ZM_1367, 2019-06-11) uses contrasts of 4% and 8%, which the paper does not describe.
- In 807 files the wheel position and its timestamps differ in length, and 16 files have no wheel
  series. In one session checked (ibl_witten_07), the wheel moves the opposite way to the choice on
  every trial.
- The archive's summary counts 178 subjects; one mouse (SWC_029) is not of the paper's project.